In [6]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/kinguistics/heartbeat-sounds/set_a_timing.csv
/kaggle/input/datasets/kinguistics/heartbeat-sounds/set_b.csv
/kaggle/input/datasets/kinguistics/heartbeat-sounds/set_a.csv
/kaggle/input/datasets/kinguistics/heartbeat-sounds/set_a/artifact__201106131834.wav
/kaggle/input/datasets/kinguistics/heartbeat-sounds/set_a/Aunlabelledtest__201106061215.wav
/kaggle/input/datasets/kinguistics/heartbeat-sounds/set_a/artifact__201106101955.wav
/kaggle/input/datasets/kinguistics/heartbeat-sounds/set_a/normal__201101151127.wav
/kaggle/input/datasets/kinguistics/heartbeat-sounds/set_a/artifact__201106220340.wav
/kaggle/input/datasets/kinguistics/heartbeat-sounds/set_a/artifact__201105190800.wav
/kaggle/input/datasets/kinguistics/heartbeat-sounds/set_a/normal__201102201230.wav
/kaggle/input/datasets/kinguistics/heartbeat-sounds/set_a/Aunlabelledtest__201108222241.wav
/kaggle/input/datasets/kinguistics/heartbeat-sounds/set_a/murmur__201102052338.wav
/kaggle/input/datasets/kinguistics

In [7]:
import os, glob, zipfile

os.system("pip install -q kaggle 2>/dev/null")
# على Kaggle، أمر kaggle datasets download يعمل عادةً بلا مصادقة للمجموعات العامة
ret = os.system("kaggle datasets download -d swapnilpanda/heart-sound-database -p /kaggle/working --unzip")
print("رمز الإرجاع:", ret)

wavs = glob.glob("/kaggle/working/**/*.wav", recursive=True)
print("عدد wav:", len(wavs))
for w in wavs[:5]:
    print("  ", w)

Dataset URL: https://www.kaggle.com/datasets/swapnilpanda/heart-sound-database
License(s): unknown


100%|██████████| 173M/173M [00:00<00:00, 233MB/s] 



رمز الإرجاع: 0
عدد wav: 3541
   /kaggle/working/heart_sound/train/healthy/e01908.wav
   /kaggle/working/heart_sound/train/healthy/e01439.wav
   /kaggle/working/heart_sound/train/healthy/a0283.wav
   /kaggle/working/heart_sound/train/healthy/e02049.wav
   /kaggle/working/heart_sound/train/healthy/a0330.wav


In [8]:
import glob, os
from collections import Counter
wavs = glob.glob("/kaggle/working/**/*.wav", recursive=True)
# المجلدات الأم (نمط التنظيم)
parents = Counter(os.path.relpath(os.path.dirname(w), "/kaggle/working") for w in wavs)
print("توزيع المجلدات:")
for k, v in sorted(parents.items()):
    print(f"  {k}: {v}")
# توزيع حرف القاعدة
letters = Counter(os.path.basename(w)[0] for w in wavs)
print("\nتوزيع حرف القاعدة:", dict(sorted(letters.items())))
# هل توجد أسماء مكررة بين train وval؟ (تسريب محتمل جاهز للتوثيق)
names = [os.path.splitext(os.path.basename(w))[0] for w in wavs]
print("\nإجمالي أسماء:", len(names), "| أسماء فريدة:", len(set(names)), "| مكررة:", len(names)-len(set(names)))

توزيع المجلدات:
  heart_sound/train/healthy: 2575
  heart_sound/train/unhealthy: 665
  heart_sound/val/healthy: 150
  heart_sound/val/unhealthy: 151

توزيع حرف القاعدة: {'a': 489, 'b': 588, 'c': 38, 'd': 65, 'e': 2247, 'f': 114}

إجمالي أسماء: 3541 | أسماء فريدة: 3240 | مكررة: 301


In [9]:
import glob, os, wave, numpy as np, random
wavs = glob.glob("/kaggle/working/**/*.wav", recursive=True)
# تسجيلات فريدة فقط (نتجاهل val المكرر)
seen, uniq = set(), []
for w in wavs:
    n = os.path.splitext(os.path.basename(w))[0]
    if n not in seen:
        seen.add(n); uniq.append(w)
random.seed(0); samp = random.sample(uniq, min(500, len(uniq)))
durs, srs = [], []
for p in samp:
    try:
        with wave.open(p,"rb") as wf:
            srs.append(wf.getframerate()); durs.append(wf.getnframes()/wf.getframerate())
    except: pass
durs=np.array(durs)
print("ترددات:", dict(zip(*np.unique(srs, return_counts=True))))
print(f"الطول (ث): أدنى={durs.min():.1f} وسيط={np.median(durs):.1f} أقصى={durs.max():.1f} متوسط={durs.mean():.1f}")
print(f"تسجيلات أقصر من 5ث: {(durs<5).sum()} | أطول من 30ث: {(durs>30).sum()}")

ترددات: {np.int64(2000): np.int64(500)}
الطول (ث): أدنى=7.0 وسيط=21.4 أقصى=99.0 متوسط=22.8
تسجيلات أقصر من 5ث: 0 | أطول من 30ث: 133


In [10]:
# =============================================================
#  المرحلة A — بناء البيانات والبروتوكولات الثلاثة
#  أصوات القلب PhysioNet 2016 — تدقيق أثر تسريب البيانات
#  تُشغّل مرة واحدة. تحفظ كل شيء في /kaggle/working/prepared
# =============================================================

import os, glob, json, hashlib
import numpy as np
import pandas as pd

RAW_ROOT = "/kaggle/working/heart_sound"   # مكان البيانات الخام (train/val/healthy/unhealthy)
OUT_DIR  = "/kaggle/working/prepared"
os.makedirs(OUT_DIR, exist_ok=True)

SR          = 2000      # تردد العيّنة (مؤكّد من الفحص)
WIN_SEC     = 3.0       # طول نافذة المقطع بالثواني
HOP_SEC     = 1.5       # الإزاحة بين المقاطع (تداخل 50%)
WIN         = int(SR * WIN_SEC)
HOP         = int(SR * HOP_SEC)
SEED        = 42
rng = np.random.default_rng(SEED)

# -------------------------------------------------------------
# 1) بناء manifest التسجيلات الفريدة
#    نتجاهل تقسيم train/val الجاهز (المصاب بالتسريب) ونعيد التجميع
# -------------------------------------------------------------
print("[1] بناء manifest التسجيلات الفريدة ...")
all_wavs = glob.glob(os.path.join(RAW_ROOT, "**", "*.wav"), recursive=True)

rows = {}
for w in all_wavs:
    rec = os.path.splitext(os.path.basename(w))[0]      # مثال a0085
    # التصنيف من اسم المجلد: healthy=0 , unhealthy=1
    parts = w.lower().split(os.sep)
    if "unhealthy" in parts:
        label = 1
    elif "healthy" in parts:
        label = 0
    else:
        continue
    db = rec[0] if rec[:1].isalpha() else "?"           # حرف القاعدة a..f
    # نحتفظ بأول ظهور فقط (التسجيلات الفريدة)
    if rec not in rows:
        rows[rec] = {"record": rec, "path": w, "db": db, "label": label}

manifest = pd.DataFrame(list(rows.values())).sort_values("record").reset_index(drop=True)
print(f"    تسجيلات فريدة: {len(manifest)}")
print(f"    توزيع التصنيف: healthy={int((manifest.label==0).sum())}  "
      f"unhealthy={int((manifest.label==1).sum())}")
print(f"    توزيع القواعد: {manifest['db'].value_counts().sort_index().to_dict()}")
manifest.to_csv(os.path.join(OUT_DIR, "manifest_records.csv"), index=False)

# -------------------------------------------------------------
# 2) التقطيع: كل تسجيل -> عدة مقاطع، مع الاحتفاظ بأصل كل مقطع
#    نحفظ المقاطع كمصفوفة float16 لتوفير المساحة + جدول فهرسة
# -------------------------------------------------------------
print("\n[2] التقطيع إلى مقاطع ...")
import wave

def read_wav(path):
    with wave.open(path, "rb") as wf:
        n = wf.getnframes()
        raw = wf.readframes(n)
    x = np.frombuffer(raw, dtype=np.int16).astype(np.float32)
    # تطبيع لكل تسجيل (z-score) — قياسي وآمن
    if x.std() > 1e-6:
        x = (x - x.mean()) / x.std()
    return x

seg_index = []   # صف لكل مقطع: seg_id, record, db, label, start
segments  = []   # الإشارة الخام للمقطع (float16)

for i, r in manifest.iterrows():
    x = read_wav(r["path"])
    if len(x) < WIN:
        # أقصر من النافذة: نكمّله بالصفر (نادر، الأدنى 6.6ث > 3ث فلا يحدث عملياً)
        x = np.pad(x, (0, WIN - len(x)))
    starts = list(range(0, len(x) - WIN + 1, HOP))
    if not starts:
        starts = [0]
    for s in starts:
        seg = x[s:s+WIN].astype(np.float16)
        seg_id = len(segments)
        segments.append(seg)
        seg_index.append({
            "seg_id": seg_id, "record": r["record"],
            "db": r["db"], "label": int(r["label"]), "start": s
        })
    if (i+1) % 500 == 0:
        print(f"    عولج {i+1}/{len(manifest)} تسجيل، مقاطع حتى الآن: {len(segments)}")

seg_df = pd.DataFrame(seg_index)
segments = np.stack(segments)   # شكل (N, WIN)
print(f"    إجمالي المقاطع: {len(seg_df)}")
print(f"    متوسط مقاطع لكل تسجيل: {len(seg_df)/len(manifest):.1f}")
print(f"    توزيع التصنيف على مستوى المقطع: "
      f"healthy={int((seg_df.label==0).sum())}  unhealthy={int((seg_df.label==1).sum())}")

np.save(os.path.join(OUT_DIR, "segments.npy"), segments)
seg_df.to_csv(os.path.join(OUT_DIR, "seg_index.csv"), index=False)
print(f"    حُفظت المقاطع: {segments.shape}, الحجم ~{segments.nbytes/1e6:.0f}MB")

# -------------------------------------------------------------
# 3) بناء البروتوكولات الثلاثة (تقسيم 70/15/15 : train/val/test)
#    كلها تستخدم نفس النِسب، الفرق فقط في وحدة التقسيم
# -------------------------------------------------------------
print("\n[3] بناء البروتوكولات الثلاثة ...")

def assign_split_by_unit(units, frac=(0.70, 0.15, 0.15), seed=SEED):
    """توزّع وحدات (مقاطع/تسجيلات/قواعد) على train/val/test."""
    units = list(units)
    r = np.random.default_rng(seed)
    r.shuffle(units)
    n = len(units)
    n_tr = int(frac[0]*n); n_va = int(frac[1]*n)
    split = {}
    for u in units[:n_tr]:            split[u] = "train"
    for u in units[n_tr:n_tr+n_va]:   split[u] = "val"
    for u in units[n_tr+n_va:]:       split[u] = "test"
    return split

protocols = {}

# (أ) المتساهل: تسريب مستوى المقطع — نخلط المقاطع نفسها
seg_ids = seg_df["seg_id"].tolist()
sp = assign_split_by_unit(seg_ids)
protocols["leaky_segment"] = seg_df["seg_id"].map(sp).tolist()

# (ب) الصارم: مستوى التسجيل — كل مقاطع التسجيل في نفس الجهة
rec_split = assign_split_by_unit(manifest["record"].tolist())
protocols["clean_record"] = seg_df["record"].map(rec_split).tolist()

# (ج) الأصرم: مستوى القاعدة — قواعد كاملة منفصلة (تعميم عبر المصادر)
#     مع 6 قواعد فقط نضع توزيعاً يدوياً متوازناً قدر الإمكان
db_list = sorted(manifest["db"].unique())
# نضمن وجود الفئتين في كل جهة قدر الإمكان؛ e ضخمة لذا تذهب للتدريب
db_split = {}
# توزيع مدروس: e (الأكبر)+a للتدريب، b للتحقق، c+d+f للاختبار
preset = {"e":"train","a":"train","b":"val","c":"test","d":"test","f":"test"}
for d in db_list:
    db_split[d] = preset.get(d, "train")
protocols["clean_database"] = seg_df["db"].map(db_split).tolist()

# نضيف الأعمدة لجدول الفهرسة ونحفظ
for name, col in protocols.items():
    seg_df[f"split_{name}"] = col
seg_df.to_csv(os.path.join(OUT_DIR, "seg_index_with_splits.csv"), index=False)

# تقرير توزيع كل بروتوكول
print("\n--- توزيع المقاطع في كل بروتوكول ---")
for name in protocols:
    col = f"split_{name}"
    print(f"\n[{name}]")
    tab = pd.crosstab(seg_df[col], seg_df["label"])
    print(tab.to_string())

# فحص حاسم: هل البروتوكول الصارم خالٍ فعلاً من تسريب التسجيل؟
print("\n--- فحص سلامة البروتوكولات ---")
def records_overlap(col):
    g = seg_df.groupby("record")[col].nunique()
    leaked = (g > 1).sum()   # تسجيل ظهر في أكثر من جهة = تسريب
    return leaked
for name in protocols:
    col = f"split_{name}"
    leaked = records_overlap(col)
    flag = "⚠️ يوجد تسريب تسجيل (متوقّع للمتساهل)" if leaked>0 else "✅ لا تسريب تسجيل"
    print(f"  {name:16s}: تسجيلات موزّعة على أكثر من جهة = {leaked}  {flag}")

# حفظ بيانات البروتوكولات
with open(os.path.join(OUT_DIR, "config.json"), "w") as f:
    json.dump({"SR":SR,"WIN_SEC":WIN_SEC,"HOP_SEC":HOP_SEC,
               "n_records":len(manifest),"n_segments":len(seg_df),
               "db_split":db_split}, f, indent=2)

print("\n✅ المرحلة A اكتملت. الملفات في /kaggle/working/prepared")
print("   - manifest_records.csv  (التسجيلات الفريدة)")
print("   - segments.npy          (المقاطع الخام)")
print("   - seg_index_with_splits.csv (الفهرسة + البروتوكولات الثلاثة)")
print("   - config.json")

[1] بناء manifest التسجيلات الفريدة ...
    تسجيلات فريدة: 3240
    توزيع التصنيف: healthy=2575  unhealthy=665
    توزيع القواعد: {'a': 409, 'b': 490, 'c': 31, 'd': 55, 'e': 2141, 'f': 114}

[2] التقطيع إلى مقاطع ...
    عولج 500/3240 تسجيل، مقاطع حتى الآن: 8581
    عولج 1000/3240 تسجيل، مقاطع حتى الآن: 11814
    عولج 1500/3240 تسجيل، مقاطع حتى الآن: 18710
    عولج 2000/3240 تسجيل، مقاطع حتى الآن: 25772
    عولج 2500/3240 تسجيل، مقاطع حتى الآن: 32698
    عولج 3000/3240 تسجيل، مقاطع حتى الآن: 39697
    إجمالي المقاطع: 43678
    متوسط مقاطع لكل تسجيل: 13.5
    توزيع التصنيف على مستوى المقطع: healthy=33367  unhealthy=10311
    حُفظت المقاطع: (43678, 6000), الحجم ~524MB

[3] بناء البروتوكولات الثلاثة ...

--- توزيع المقاطع في كل بروتوكول ---

[leaky_segment]
label                    0     1
split_leaky_segment             
test                  5016  1537
train                23351  7223
val                   5000  1551

[clean_record]
label                   0     1
split_clean_record    

In [11]:
# =============================================================
#  المرحلة B (نسخة CPU سريعة) — التدريب وقياس فجوة التضخّم
#  التحسين الأساسي: حساب mel-spectrogram مسبقاً مرة واحدة
#  بدل إعادة حسابه لكل مقطع في كل epoch (القاتل الحقيقي للسرعة)
# =============================================================

import os, json, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, f1_score, balanced_accuracy_score, accuracy_score

PREP = "/kaggle/working/prepared"
OUT  = "/kaggle/working/results"
os.makedirs(OUT, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("الجهاز:", device)
torch.set_num_threads(os.cpu_count())   # استغلال كل أنوية CPU

SR        = 2000
N_SEEDS   = 3
SEEDS     = [42, 123, 2024]
EPOCHS    = 8
PATIENCE  = 3
BATCH     = 256
N_MELS    = 40
PROTOCOLS = ["leaky_segment", "clean_record", "clean_database"]

# -------------------------------------------------------------
# 1) تحميل المقاطع
# -------------------------------------------------------------
print("\n[1] تحميل المقاطع ...")
segments = np.load(os.path.join(PREP, "segments.npy"))
seg_df   = pd.read_csv(os.path.join(PREP, "seg_index_with_splits.csv"))
print(f"    مقاطع: {segments.shape}")

# -------------------------------------------------------------
# 2) حساب mel-spectrogram مسبقاً لكل المقاطع (مرة واحدة)
#    هذا أهم تحسين — يحوّل ساعات إلى دقائق على CPU
# -------------------------------------------------------------
MEL_CACHE = os.path.join(PREP, "mels.npy")
if os.path.exists(MEL_CACHE):
    print("\n[2] تحميل mel-spectrograms المحفوظة مسبقاً ...")
    mels = np.load(MEL_CACHE)
else:
    print("\n[2] حساب mel-spectrograms مسبقاً (مرة واحدة) ...")
    import torchaudio
    mel_tf = torchaudio.transforms.MelSpectrogram(
        sample_rate=SR, n_fft=256, hop_length=64, n_mels=N_MELS, power=2.0)
    db_tf = torchaudio.transforms.AmplitudeToDB(stype="power")

    mels_list = []
    B = 512
    t0 = time.time()
    for i in range(0, len(segments), B):
        chunk = torch.from_numpy(segments[i:i+B].astype(np.float32))
        m = db_tf(mel_tf(chunk))                       # (b, n_mels, time)
        m = (m - m.mean(dim=(1,2), keepdim=True)) / (m.std(dim=(1,2), keepdim=True) + 1e-6)
        mels_list.append(m.numpy().astype(np.float16))
        if (i//B) % 20 == 0:
            print(f"    {i}/{len(segments)} ... ({time.time()-t0:.0f}s)")
    mels = np.concatenate(mels_list)
    np.save(MEL_CACHE, mels)
    print(f"    اكتمل الحساب في {(time.time()-t0)/60:.1f} دقيقة | الشكل: {mels.shape}")

MEL_T = mels.shape[-1]
print(f"    أبعاد mel: ({N_MELS}, {MEL_T})")

# -------------------------------------------------------------
# 3) Dataset يقرأ mel الجاهز (سريع جداً)
# -------------------------------------------------------------
class MelDataset(Dataset):
    def __init__(self, idx):
        self.sid = idx["seg_id"].values
        self.y   = idx["label"].values.astype(np.float32)
    def __len__(self): return len(self.sid)
    def __getitem__(self, i):
        s = self.sid[i]
        x = torch.from_numpy(mels[s].astype(np.float32)).unsqueeze(0)  # (1,n_mels,T)
        return x, self.y[i], int(s)

# -------------------------------------------------------------
# 4) نموذج CNN خفيف
# -------------------------------------------------------------
class LightCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1,16,3,padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1))
        self.fc = nn.Linear(64,1)
    def forward(self,x): return self.fc(self.net(x).flatten(1)).squeeze(1)

def n_params(m): return sum(p.numel() for p in m.parameters())

# -------------------------------------------------------------
# 5) التقييم (مستوى التسجيل + مستوى المقطع)
# -------------------------------------------------------------
def predict(model, loader):
    model.eval(); P, Y, S = [], [], []
    with torch.no_grad():
        for xb, yb, sid in loader:
            xb = xb.to(device)
            P.append(torch.sigmoid(model(xb)).cpu().numpy())
            Y.append(yb.numpy()); S.append(sid.numpy())
    return np.concatenate(P), np.concatenate(Y), np.concatenate(S)

def metrics_from(y, p):
    yh = (p>=0.5).astype(int)
    return {"auc": roc_auc_score(y,p) if len(np.unique(y))>1 else float("nan"),
            "f1": f1_score(y,yh,zero_division=0),
            "bal_acc": balanced_accuracy_score(y,yh),
            "acc": accuracy_score(y,yh)}

def eval_both(model, loader, idx_df):
    p, y, s = predict(model, loader)
    seg_m = metrics_from(y, p)                      # مستوى المقطع
    tmp = pd.DataFrame({"seg_id":s,"prob":p}).merge(
        idx_df[["seg_id","record","label"]], on="seg_id")
    rec = tmp.groupby("record").agg(prob=("prob","mean"),
                                    label=("label","first")).reset_index()
    rec_m = metrics_from(rec["label"].values, rec["prob"].values)
    return rec_m, seg_m

def set_seed(s):
    np.random.seed(s); torch.manual_seed(s)

# -------------------------------------------------------------
# 6) تدريب واحد
# -------------------------------------------------------------
def run_one(protocol, seed):
    set_seed(seed)
    col = f"split_{protocol}"
    tr = seg_df[seg_df[col]=="train"]; va = seg_df[seg_df[col]=="val"]; te = seg_df[seg_df[col]=="test"]
    trL = DataLoader(MelDataset(tr), batch_size=BATCH, shuffle=True,  num_workers=2)
    vaL = DataLoader(MelDataset(va), batch_size=BATCH, shuffle=False, num_workers=2)
    teL = DataLoader(MelDataset(te), batch_size=BATCH, shuffle=False, num_workers=2)

    model = LightCNN().to(device)
    pos_w = torch.tensor([(tr["label"]==0).sum()/max(1,(tr["label"]==1).sum())], device=device)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    opt  = torch.optim.Adam(model.parameters(), lr=1e-3)

    best_auc, best_state, bad = -1, None, 0
    for ep in range(EPOCHS):
        model.train()
        for xb, yb, _ in trL:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
        vm,_ = eval_both(model, vaL, seg_df)
        if not np.isnan(vm["auc"]) and vm["auc"]>best_auc:
            best_auc=vm["auc"]; best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}; bad=0
        else:
            bad+=1
            if bad>=PATIENCE: break
    if best_state: model.load_state_dict(best_state)
    rec_m, seg_m = eval_both(model, teL, seg_df)
    return rec_m, seg_m, n_params(model)

# -------------------------------------------------------------
# 7) التشغيل
# -------------------------------------------------------------
print("\n[7] بدء التدريب ...\n")
rows=[]; t0=time.time()
for proto in PROTOCOLS:
    for sd in SEEDS:
        t1=time.time(); rec_m, seg_m, npar = run_one(proto, sd); dt=time.time()-t1
        rows.append({"protocol":proto,"seed":sd,"eval":"record","params":npar,**rec_m})
        rows.append({"protocol":proto,"seed":sd,"eval":"segment","params":npar,**seg_m})
        print(f"  {proto:16s} seed={sd:<5d} REC auc={rec_m['auc']:.3f} f1={rec_m['f1']:.3f} "
              f"bal={rec_m['bal_acc']:.3f} | SEG auc={seg_m['auc']:.3f}  ({dt:.0f}s)")

res = pd.DataFrame(rows); res.to_csv(os.path.join(OUT,"raw_results.csv"), index=False)
print(f"\nالزمن الكلي: {(time.time()-t0)/60:.1f} دقيقة")

# -------------------------------------------------------------
# 8) الملخص وفجوة التضخّم
# -------------------------------------------------------------
print("\n"+"="*70+"\nالملخص: المتوسط ± الانحراف المعياري\n"+"="*70)
for ev in ["record","segment"]:
    print(f"\n--- مستوى: {ev} ---")
    sub=res[res["eval"]==ev]
    g=sub.groupby("protocol").agg(auc_m=("auc","mean"),auc_s=("auc","std"),
        f1_m=("f1","mean"),f1_s=("f1","std"),bal_m=("bal_acc","mean"),bal_s=("bal_acc","std"))
    for p in PROTOCOLS:
        if p in g.index:
            r=g.loc[p]
            print(f"  {p:16s}: AUC={r.auc_m:.3f}±{r.auc_s:.3f}  F1={r.f1_m:.3f}±{r.f1_s:.3f}  bal={r.bal_m:.3f}±{r.bal_s:.3f}")

print("\n"+"="*70+"\nفجوة التضخّم (رسالة الورقة)\n"+"="*70)
rec=res[res["eval"]=="record"].groupby("protocol")["auc"].mean()
seg=res[res["eval"]=="segment"].groupby("protocol")["auc"].mean()
print(f"  AUC مُبلّغ (متساهل، مستوى المقطع): {seg['leaky_segment']:.3f}")
print(f"  AUC حقيقي (صارم، مستوى التسجيل) : {rec['clean_record']:.3f}")
print(f"  >>> فجوة التضخّم = {seg['leaky_segment']-rec['clean_record']:+.3f} AUC")
print(f"  AUC تعميم عبر القواعد (الأصعب)  : {rec['clean_database']:.3f}")
print("\n✅ اكتملت. النتائج محفوظة في results/raw_results.csv")

الجهاز: cuda

[1] تحميل المقاطع ...
    مقاطع: (43678, 6000)

[2] حساب mel-spectrograms مسبقاً (مرة واحدة) ...
    0/43678 ... (0s)
    10240/43678 ... (2s)
    20480/43678 ... (4s)
    30720/43678 ... (6s)
    40960/43678 ... (9s)
    اكتمل الحساب في 0.2 دقيقة | الشكل: (43678, 40, 94)
    أبعاد mel: (40, 94)

[7] بدء التدريب ...

  leaky_segment    seed=42    REC auc=0.977 f1=0.740 bal=0.902 | SEG auc=0.977  (36s)
  leaky_segment    seed=123   REC auc=0.979 f1=0.845 bal=0.904 | SEG auc=0.980  (32s)
  leaky_segment    seed=2024  REC auc=0.979 f1=0.803 bal=0.923 | SEG auc=0.979  (32s)
  clean_record     seed=42    REC auc=0.968 f1=0.588 bal=0.820 | SEG auc=0.973  (33s)
  clean_record     seed=123   REC auc=0.967 f1=0.807 bal=0.917 | SEG auc=0.973  (33s)
  clean_record     seed=2024  REC auc=0.965 f1=0.646 bal=0.747 | SEG auc=0.971  (33s)
  clean_database   seed=42    REC auc=0.434 f1=0.504 bal=0.438 | SEG auc=0.394  (37s)
  clean_database   seed=123   REC auc=0.396 f1=0.594 bal=0.493 | 

In [12]:
# =============================================================
#  المرحلة C — تجربة الضبط (Control)
#  السؤال: انهيار clean_database (0.97 -> 0.39) — سببه التسريب
#  عبر القواعد، أم اختلاف توزيع الفئات بين التدريب والاختبار؟
#  نفصل العاملين. يبني على mels المحفوظة (سريع).
# =============================================================

import os, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, f1_score

PREP = "/kaggle/working/prepared"
OUT  = "/kaggle/working/results"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("الجهاز:", device)
torch.set_num_threads(os.cpu_count())

SEEDS  = [42, 123, 2024]
EPOCHS = 8; PATIENCE = 3; BATCH = 256; N_MELS = 40

print("\n[1] تحميل mels والفهرسة المحفوظة ...")
mels   = np.load(os.path.join(PREP, "mels.npy"))
seg_df = pd.read_csv(os.path.join(PREP, "seg_index_with_splits.csv"))

# -------------------------------------------------------------
# أولاً: تشخيص توزيع الفئات في كل قاعدة (لفهم المشكلة)
# -------------------------------------------------------------
print("\n[2] توزيع الفئات لكل قاعدة (على مستوى التسجيل):")
rec_level = seg_df.drop_duplicates("record")[["record","db","label"]]
tab = pd.crosstab(rec_level["db"], rec_level["label"])
tab.columns = ["healthy","unhealthy"]
tab["%unhealthy"] = (100*tab["unhealthy"]/(tab["healthy"]+tab["unhealthy"])).round(1)
print(tab.to_string())
print("\nملاحظة: قواعد التدريب (e,a) مقابل الاختبار (c,d,f) — قارني %unhealthy")

# -------------------------------------------------------------
# نموذج وأدوات (نفس المرحلة B)
# -------------------------------------------------------------
class MelDataset(Dataset):
    def __init__(self, idx):
        self.sid = idx["seg_id"].values; self.y = idx["label"].values.astype(np.float32)
    def __len__(self): return len(self.sid)
    def __getitem__(self, i):
        s=self.sid[i]
        return torch.from_numpy(mels[s].astype(np.float32)).unsqueeze(0), self.y[i], int(s)

class LightCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(1,16,3,padding=1),nn.BatchNorm2d(16),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),nn.AdaptiveAvgPool2d(1))
        self.fc=nn.Linear(64,1)
    def forward(self,x): return self.fc(self.net(x).flatten(1)).squeeze(1)

def predict(model, loader):
    model.eval(); P,Y,S=[],[],[]
    with torch.no_grad():
        for xb,yb,sid in loader:
            P.append(torch.sigmoid(model(xb.to(device))).cpu().numpy())
            Y.append(yb.numpy()); S.append(sid.numpy())
    return np.concatenate(P),np.concatenate(Y),np.concatenate(S)

def rec_auc(model, loader, idx_df):
    p,y,s=predict(model,loader)
    tmp=pd.DataFrame({"seg_id":s,"prob":p}).merge(idx_df[["seg_id","record","label"]],on="seg_id")
    rec=tmp.groupby("record").agg(prob=("prob","mean"),label=("label","first")).reset_index()
    yv,pv=rec["label"].values,rec["prob"].values
    if len(np.unique(yv))<2: return float("nan"),float("nan")
    return roc_auc_score(yv,pv), balanced_accuracy_score(yv,(pv>=0.5).astype(int))

def set_seed(s): np.random.seed(s); torch.manual_seed(s)

def train_eval(tr, va, te, seed, balance_train=False):
    set_seed(seed)
    if balance_train:
        # موازنة: نأخذ عيّنة متساوية من الفئتين على مستوى التسجيل
        recs = tr.drop_duplicates("record")[["record","label"]]
        n_min = recs["label"].value_counts().min()
        keep = pd.concat([
            recs[recs.label==0].sample(n_min, random_state=seed),
            recs[recs.label==1].sample(n_min, random_state=seed)])["record"]
        tr = tr[tr["record"].isin(keep)]
    trL=DataLoader(MelDataset(tr),batch_size=BATCH,shuffle=True,num_workers=2)
    vaL=DataLoader(MelDataset(va),batch_size=BATCH,shuffle=False,num_workers=2)
    teL=DataLoader(MelDataset(te),batch_size=BATCH,shuffle=False,num_workers=2)
    model=LightCNN().to(device)
    pos_w=torch.tensor([(tr.label==0).sum()/max(1,(tr.label==1).sum())],device=device)
    crit=nn.BCEWithLogitsLoss(pos_weight=pos_w); opt=torch.optim.Adam(model.parameters(),1e-3)
    best,bs,bad=-1,None,0
    for ep in range(EPOCHS):
        model.train()
        for xb,yb,_ in trL:
            opt.zero_grad(); crit(model(xb.to(device)),yb.to(device)).backward(); opt.step()
        a,_=rec_auc(model,vaL,seg_df)
        if not np.isnan(a) and a>best: best,bs,bad=a,{k:v.cpu().clone() for k,v in model.state_dict().items()},0
        else:
            bad+=1
            if bad>=PATIENCE: break
    if bs: model.load_state_dict(bs)
    return rec_auc(model,teL,seg_df)

col="split_clean_database"
tr=seg_df[seg_df[col]=="train"]; va=seg_df[seg_df[col]=="val"]; te=seg_df[seg_df[col]=="test"]

# -------------------------------------------------------------
# التجربة: clean_database أصلي مقابل نسخة بتدريب متوازن
# -------------------------------------------------------------
print("\n[3] تشغيل المقارنة (أصلي مقابل تدريب متوازن) ...\n")
res=[]
for tag, bal in [("original", False), ("balanced_train", True)]:
    for sd in SEEDS:
        t1=time.time(); auc,bal_acc=train_eval(tr,va,te,sd,balance_train=bal); dt=time.time()-t1
        res.append({"setting":tag,"seed":sd,"auc":auc,"bal_acc":bal_acc})
        print(f"  {tag:16s} seed={sd:<5d} AUC={auc:.3f} balAcc={bal_acc:.3f}  ({dt:.0f}s)")

r=pd.DataFrame(res)
print("\n"+"="*60+"\nالنتيجة\n"+"="*60)
g=r.groupby("setting").agg(auc_m=("auc","mean"),auc_s=("auc","std"),
                            bal_m=("bal_acc","mean"),bal_s=("bal_acc","std"))
for s in ["original","balanced_train"]:
    if s in g.index:
        x=g.loc[s]; print(f"  {s:16s}: AUC={x.auc_m:.3f}±{x.auc_s:.3f}  balAcc={x.bal_m:.3f}±{x.bal_s:.3f}")

print("""
التفسير:
- لو بقي AUC منخفضاً (<0.6) حتى مع تدريب متوازن:
  => الانهيار سببه تسريب المصدر/القاعدة الحقيقي. رسالتنا قوية ✅
- لو ارتفع AUC كثيراً مع الموازنة:
  => جزء كبير من الانهيار كان اختلاف توزيع الفئات، نعدّل الرسالة.
""")
r.to_csv(os.path.join(OUT,"control_results.csv"),index=False)
print("✅ حُفظت في results/control_results.csv")

الجهاز: cuda

[1] تحميل mels والفهرسة المحفوظة ...

[2] توزيع الفئات لكل قاعدة (على مستوى التسجيل):
    healthy  unhealthy  %unhealthy
db                                
a       117        292        71.4
b       386        104        21.2
c         7         24        77.4
d        27         28        50.9
e      1958        183         8.5
f        80         34        29.8

ملاحظة: قواعد التدريب (e,a) مقابل الاختبار (c,d,f) — قارني %unhealthy

[3] تشغيل المقارنة (أصلي مقابل تدريب متوازن) ...

  original         seed=42    AUC=0.435 balAcc=0.429  (37s)
  original         seed=123   AUC=0.385 balAcc=0.499  (28s)
  original         seed=2024  AUC=0.417 balAcc=0.491  (32s)
  balanced_train   seed=42    AUC=0.408 balAcc=0.423  (17s)
  balanced_train   seed=123   AUC=0.384 balAcc=0.443  (17s)
  balanced_train   seed=2024  AUC=0.404 balAcc=0.493  (17s)

النتيجة
  original        : AUC=0.412±0.025  balAcc=0.473±0.038
  balanced_train  : AUC=0.398±0.013  balAcc=0.453±0.036

التفسير:
- لو بق

In [13]:
# =============================================================
#  المرحلة D (النهائية) — 5 seeds + اختبار دلالة إحصائية
#  • يتحقق تلقائياً من وجود البيانات المجهّزة، ويعيد بناءها إن لزم
#  • يحفظ النتائج تدريجياً بعد كل تدريب (حماية من انقطاع الجلسة)
#  • يُجري paired t-test و Wilcoxon بين البروتوكولات
# =============================================================

import os, time, glob, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, f1_score, accuracy_score
from scipy import stats

PREP = "/kaggle/working/prepared"
OUT  = "/kaggle/working/results"
os.makedirs(OUT, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("الجهاز:", device)
torch.set_num_threads(os.cpu_count())

SR=2000; WIN=int(SR*3.0); HOP=int(SR*1.5)
N_MELS=40; EPOCHS=8; PATIENCE=3; BATCH=256
SEEDS=[42,123,2024,7,99]                         # 5 seeds
PROTOCOLS=["leaky_segment","clean_record","clean_database"]

# -------------------------------------------------------------
# 0) التأكد من وجود البيانات المجهّزة — وإعادة بنائها إن مُسحت
# -------------------------------------------------------------
need_rebuild = not (os.path.exists(os.path.join(PREP,"mels.npy")) and
                    os.path.exists(os.path.join(PREP,"seg_index_with_splits.csv")))
if need_rebuild:
    print("\n⚠️ البيانات المجهّزة غير موجودة — إعادة البناء ...")
    # تأكدي أن البيانات الخام موجودة، وإلا حمّليها أولاً
    raw = glob.glob("/kaggle/working/heart_sound/**/*.wav", recursive=True)
    if len(raw)==0:
        print("تحميل البيانات الخام من Kaggle ...")
        os.system("kaggle datasets download -d swapnilpanda/heart-sound-database -p /kaggle/working --unzip")
        raw = glob.glob("/kaggle/working/heart_sound/**/*.wav", recursive=True)
    print(f"ملفات wav الخام: {len(raw)}")

    import wave
    os.makedirs(PREP, exist_ok=True)
    rng=np.random.default_rng(42)
    # manifest
    rows={}
    for w in raw:
        rec=os.path.splitext(os.path.basename(w))[0]
        p=w.lower().split(os.sep)
        lab=1 if "unhealthy" in p else (0 if "healthy" in p else None)
        if lab is None: continue
        db=rec[0] if rec[:1].isalpha() else "?"
        if rec not in rows: rows[rec]={"record":rec,"path":w,"db":db,"label":lab}
    man=pd.DataFrame(list(rows.values())).sort_values("record").reset_index(drop=True)
    # تقطيع
    def rd(path):
        with wave.open(path,"rb") as wf: x=np.frombuffer(wf.readframes(wf.getnframes()),dtype=np.int16).astype(np.float32)
        return (x-x.mean())/(x.std()+1e-6) if x.std()>1e-6 else x
    segs=[]; idx=[]
    for _,r in man.iterrows():
        x=rd(r["path"])
        if len(x)<WIN: x=np.pad(x,(0,WIN-len(x)))
        starts=list(range(0,len(x)-WIN+1,HOP)) or [0]
        for s in starts:
            idx.append({"seg_id":len(segs),"record":r["record"],"db":r["db"],"label":int(r["label"])})
            segs.append(x[s:s+WIN].astype(np.float16))
    segs=np.stack(segs); sdf=pd.DataFrame(idx)
    # بروتوكولات
    def sp(units,frac=(.7,.15,.15),seed=42):
        u=list(units); np.random.default_rng(seed).shuffle(u); n=len(u)
        a=int(frac[0]*n); b=int(frac[1]*n); d={}
        for x in u[:a]:d[x]="train"
        for x in u[a:a+b]:d[x]="val"
        for x in u[a+b:]:d[x]="test"
        return d
    sdf["split_leaky_segment"]=sdf["seg_id"].map(sp(sdf["seg_id"].tolist()))
    sdf["split_clean_record"]=sdf["record"].map(sp(man["record"].tolist()))
    preset={"e":"train","a":"train","b":"val","c":"test","d":"test","f":"test"}
    sdf["split_clean_database"]=sdf["db"].map(lambda d:preset.get(d,"train"))
    # mels
    import torchaudio
    mt=torchaudio.transforms.MelSpectrogram(sample_rate=SR,n_fft=256,hop_length=64,n_mels=N_MELS,power=2.0)
    dt=torchaudio.transforms.AmplitudeToDB(stype="power")
    ms=[]
    for i in range(0,len(segs),512):
        m=dt(mt(torch.from_numpy(segs[i:i+512].astype(np.float32))))
        m=(m-m.mean(dim=(1,2),keepdim=True))/(m.std(dim=(1,2),keepdim=True)+1e-6)
        ms.append(m.numpy().astype(np.float16))
    mels=np.concatenate(ms)
    np.save(os.path.join(PREP,"mels.npy"),mels)
    sdf.to_csv(os.path.join(PREP,"seg_index_with_splits.csv"),index=False)
    print("✅ أُعيد البناء.")

print("\n[1] تحميل البيانات المجهّزة ...")
mels=np.load(os.path.join(PREP,"mels.npy"))
seg_df=pd.read_csv(os.path.join(PREP,"seg_index_with_splits.csv"))
print(f"    mels: {mels.shape}")

# -------------------------------------------------------------
# النموذج والأدوات
# -------------------------------------------------------------
class MelDS(Dataset):
    def __init__(s,idx): s.sid=idx["seg_id"].values; s.y=idx["label"].values.astype(np.float32)
    def __len__(s): return len(s.sid)
    def __getitem__(s,i):
        j=s.sid[i]; return torch.from_numpy(mels[j].astype(np.float32)).unsqueeze(0),s.y[i],int(j)

class LightCNN(nn.Module):
    def __init__(s):
        super().__init__()
        s.net=nn.Sequential(
            nn.Conv2d(1,16,3,padding=1),nn.BatchNorm2d(16),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),nn.AdaptiveAvgPool2d(1))
        s.fc=nn.Linear(64,1)
    def forward(s,x): return s.fc(s.net(x).flatten(1)).squeeze(1)

def predict(model,loader):
    model.eval(); P,Y,S=[],[],[]
    with torch.no_grad():
        for xb,yb,sid in loader:
            P.append(torch.sigmoid(model(xb.to(device))).cpu().numpy()); Y.append(yb.numpy()); S.append(sid.numpy())
    return np.concatenate(P),np.concatenate(Y),np.concatenate(S)

def metr(y,p):
    yh=(p>=.5).astype(int)
    return (roc_auc_score(y,p) if len(np.unique(y))>1 else np.nan,
            f1_score(y,yh,zero_division=0),balanced_accuracy_score(y,yh),accuracy_score(y,yh))

def eval_both(model,loader):
    p,y,s=predict(model,loader)
    seg=metr(y,p)
    t=pd.DataFrame({"seg_id":s,"prob":p}).merge(seg_df[["seg_id","record","label"]],on="seg_id")
    rec=t.groupby("record").agg(prob=("prob","mean"),label=("label","first")).reset_index()
    return metr(rec["label"].values,rec["prob"].values),seg

def run(proto,seed):
    np.random.seed(seed); torch.manual_seed(seed)
    c=f"split_{proto}"
    tr=seg_df[seg_df[c]=="train"]; va=seg_df[seg_df[c]=="val"]; te=seg_df[seg_df[c]=="test"]
    trL=DataLoader(MelDS(tr),BATCH,shuffle=True,num_workers=2)
    vaL=DataLoader(MelDS(va),BATCH,shuffle=False,num_workers=2)
    teL=DataLoader(MelDS(te),BATCH,shuffle=False,num_workers=2)
    m=LightCNN().to(device)
    pw=torch.tensor([(tr.label==0).sum()/max(1,(tr.label==1).sum())],device=device)
    crit=nn.BCEWithLogitsLoss(pos_weight=pw); opt=torch.optim.Adam(m.parameters(),1e-3)
    best,bs,bad=-1,None,0
    for ep in range(EPOCHS):
        m.train()
        for xb,yb,_ in trL:
            opt.zero_grad(); crit(m(xb.to(device)),yb.to(device)).backward(); opt.step()
        (va_auc,_,_,_),_=eval_both(m,vaL)
        if not np.isnan(va_auc) and va_auc>best: best,bs,bad=va_auc,{k:v.cpu().clone() for k,v in m.state_dict().items()},0
        else:
            bad+=1
            if bad>=PATIENCE: break
    if bs: m.load_state_dict(bs)
    return eval_both(m,teL)

# -------------------------------------------------------------
# التشغيل مع حفظ تدريجي
# -------------------------------------------------------------
print("\n[2] التدريب (5 seeds × 3 بروتوكولات) — حفظ تدريجي ...\n")
prog_path=os.path.join(OUT,"final_results.csv")
done=set()
rows=[]
if os.path.exists(prog_path):
    prev=pd.read_csv(prog_path); rows=prev.to_dict("records")
    done={(r["protocol"],r["seed"],r["eval"]) for r in rows}
    print(f"    استئناف: {len(done)} نتيجة محفوظة مسبقاً.")

t0=time.time()
for proto in PROTOCOLS:
    for sd in SEEDS:
        if (proto,sd,"record") in done:
            print(f"  تخطّي {proto} seed={sd} (محفوظ)"); continue
        t1=time.time(); (ra,rf,rb,rac),(sa,sf,sb,sac)=run(proto,sd); dt=time.time()-t1
        rows.append({"protocol":proto,"seed":sd,"eval":"record","auc":ra,"f1":rf,"bal_acc":rb,"acc":rac})
        rows.append({"protocol":proto,"seed":sd,"eval":"segment","auc":sa,"f1":sf,"bal_acc":sb,"acc":sac})
        pd.DataFrame(rows).to_csv(prog_path,index=False)   # حفظ فوري
        print(f"  {proto:16s} seed={sd:<4d} REC auc={ra:.3f} bal={rb:.3f} | SEG auc={sa:.3f}  ({dt:.0f}s) [حُفظ]")
print(f"\nالزمن الكلي: {(time.time()-t0)/60:.1f} دقيقة")

res=pd.DataFrame(rows)

# -------------------------------------------------------------
# الملخص
# -------------------------------------------------------------
print("\n"+"="*70+"\nالملخص النهائي (5 seeds): المتوسط ± الانحراف\n"+"="*70)
for ev in ["record","segment"]:
    print(f"\n--- مستوى {ev} ---")
    g=res[res.eval==ev].groupby("protocol").agg(
        auc_m=("auc","mean"),auc_s=("auc","std"),bal_m=("bal_acc","mean"),bal_s=("bal_acc","std"),
        f1_m=("f1","mean"),f1_s=("f1","std"))
    for p in PROTOCOLS:
        if p in g.index:
            r=g.loc[p]; print(f"  {p:16s}: AUC={r.auc_m:.3f}±{r.auc_s:.3f}  bal={r.bal_m:.3f}±{r.bal_s:.3f}  F1={r.f1_m:.3f}±{r.f1_s:.3f}")

# -------------------------------------------------------------
# اختبار الدلالة الإحصائية (مستوى التسجيل، AUC)
# -------------------------------------------------------------
print("\n"+"="*70+"\nاختبار الدلالة الإحصائية (AUC، مستوى التسجيل)\n"+"="*70)
def aucs(p): return res[(res.protocol==p)&(res.eval=="record")].sort_values("seed")["auc"].values
pairs=[("leaky_segment","clean_record"),("clean_record","clean_database"),("leaky_segment","clean_database")]
for a,b in pairs:
    x,y=aucs(a),aucs(b)
    t_t,t_p=stats.ttest_rel(x,y)
    try: w_s,w_p=stats.wilcoxon(x,y)
    except Exception: w_p=np.nan
    print(f"  {a} مقابل {b}:")
    print(f"     فرق المتوسط = {x.mean()-y.mean():+.3f} | paired t p={t_p:.4f} | Wilcoxon p={w_p:.4f}")

print("\n✅ اكتمل. النتائج النهائية في results/final_results.csv")

الجهاز: cuda

[1] تحميل البيانات المجهّزة ...
    mels: (43678, 40, 94)

[2] التدريب (5 seeds × 3 بروتوكولات) — حفظ تدريجي ...

  leaky_segment    seed=42   REC auc=0.977 bal=0.909 | SEG auc=0.978  (32s) [حُفظ]
  leaky_segment    seed=123  REC auc=0.978 bal=0.915 | SEG auc=0.979  (32s) [حُفظ]
  leaky_segment    seed=2024 REC auc=0.979 bal=0.922 | SEG auc=0.979  (32s) [حُفظ]
  leaky_segment    seed=7    REC auc=0.976 bal=0.922 | SEG auc=0.975  (32s) [حُفظ]
  leaky_segment    seed=99   REC auc=0.978 bal=0.905 | SEG auc=0.978  (32s) [حُفظ]
  clean_record     seed=42   REC auc=0.968 bal=0.839 | SEG auc=0.973  (32s) [حُفظ]
  clean_record     seed=123  REC auc=0.967 bal=0.912 | SEG auc=0.972  (32s) [حُفظ]
  clean_record     seed=2024 REC auc=0.963 bal=0.732 | SEG auc=0.970  (32s) [حُفظ]
  clean_record     seed=7    REC auc=0.967 bal=0.762 | SEG auc=0.973  (32s) [حُفظ]
  clean_record     seed=99   REC auc=0.956 bal=0.886 | SEG auc=0.967  (24s) [حُفظ]
  clean_database   seed=42   REC auc=0.435

KeyError: False

In [14]:
import pandas as pd, numpy as np
from scipy import stats

res = pd.read_csv("/kaggle/working/results/final_results.csv")
print("عدد النتائج المحفوظة:", len(res), "| seeds مكتملة:")
print(res[res["eval"]=="record"].groupby("protocol")["seed"].apply(list))

PROTOCOLS=["leaky_segment","clean_record","clean_database"]
print("\n"+"="*70+"\nالملخص النهائي: المتوسط ± الانحراف\n"+"="*70)
for ev in ["record","segment"]:
    print(f"\n--- مستوى {ev} ---")
    sub = res[res["eval"]==ev]
    g = sub.groupby("protocol").agg(
        auc_m=("auc","mean"), auc_s=("auc","std"),
        bal_m=("bal_acc","mean"), bal_s=("bal_acc","std"),
        f1_m=("f1","mean"), f1_s=("f1","std"))
    for p in PROTOCOLS:
        if p in g.index:
            r = g.loc[p]
            print(f"  {p:16s}: AUC={r.auc_m:.3f}±{r.auc_s:.3f}  "
                  f"bal={r.bal_m:.3f}±{r.bal_s:.3f}  F1={r.f1_m:.3f}±{r.f1_s:.3f}")

print("\n"+"="*70+"\nاختبار الدلالة (AUC، مستوى التسجيل)\n"+"="*70)
def aucs(p): return res[(res["protocol"]==p)&(res["eval"]=="record")].sort_values("seed")["auc"].values
for a,b in [("leaky_segment","clean_record"),("clean_record","clean_database"),("leaky_segment","clean_database")]:
    x,y = aucs(a), aucs(b)
    if len(x)==len(y) and len(x)>=2:
        t_p = stats.ttest_rel(x,y).pvalue
        try: w_p = stats.wilcoxon(x,y).pvalue
        except: w_p = float("nan")
        print(f"  {a} مقابل {b}: فرق={x.mean()-y.mean():+.3f}  t_p={t_p:.4f}  Wilcoxon_p={w_p:.4f}")

عدد النتائج المحفوظة: 30 | seeds مكتملة:
protocol
clean_database    [42, 123, 2024, 7, 99]
clean_record      [42, 123, 2024, 7, 99]
leaky_segment     [42, 123, 2024, 7, 99]
Name: seed, dtype: object

الملخص النهائي: المتوسط ± الانحراف

--- مستوى record ---
  leaky_segment   : AUC=0.977±0.001  bal=0.914±0.007  F1=0.792±0.040
  clean_record    : AUC=0.964±0.005  bal=0.826±0.077  F1=0.693±0.084
  clean_database  : AUC=0.395±0.087  bal=0.450±0.088  F1=0.523±0.133

--- مستوى segment ---
  leaky_segment   : AUC=0.978±0.002  bal=0.921±0.006  F1=0.818±0.030
  clean_record    : AUC=0.971±0.003  bal=0.845±0.081  F1=0.732±0.086
  clean_database  : AUC=0.397±0.079  bal=0.449±0.073  F1=0.558±0.127

اختبار الدلالة (AUC، مستوى التسجيل)
  leaky_segment مقابل clean_record: فرق=+0.013  t_p=0.0050  Wilcoxon_p=0.0625
  clean_record مقابل clean_database: فرق=+0.569  t_p=0.0001  Wilcoxon_p=0.0625
  leaky_segment مقابل clean_database: فرق=+0.583  t_p=0.0001  Wilcoxon_p=0.0625


In [15]:
# =============================================================
#  المرحلة E — الحل: Domain-Adversarial Training (DANN)
#  الهدف: رفع AUC عبر القواعد (الأساس ~0.40) بجعل النموذج
#  يتعلّم سمات لا تميّز المصدر/القاعدة -> يركّز على المرض
#  يقارن: baseline مقابل DANN على البروتوكول clean_database
# =============================================================

import os, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.autograd import Function
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, f1_score

PREP="/kaggle/working/prepared"; OUT="/kaggle/working/results"
device="cuda" if torch.cuda.is_available() else "cpu"
print("الجهاز:", device); torch.set_num_threads(os.cpu_count())

SEEDS=[42,123,2024,7,99]; EPOCHS=10; PATIENCE=4; BATCH=256; N_MELS=40

print("\n[1] تحميل البيانات ...")
mels=np.load(os.path.join(PREP,"mels.npy"))
seg_df=pd.read_csv(os.path.join(PREP,"seg_index_with_splits.csv"))

# ترميز القاعدة كرقم (لرأس التنبؤ بالمصدر)
dbs=sorted(seg_df["db"].unique())
db2i={d:i for i,d in enumerate(dbs)}
seg_df["db_idx"]=seg_df["db"].map(db2i)
N_DB=len(dbs)
print(f"    القواعد: {dbs} (عددها {N_DB})")

class DS(Dataset):
    def __init__(s,idx):
        s.sid=idx["seg_id"].values; s.y=idx["label"].values.astype(np.float32)
        s.d=idx["db_idx"].values.astype(np.int64)
    def __len__(s): return len(s.sid)
    def __getitem__(s,i):
        j=s.sid[i]
        return (torch.from_numpy(mels[j].astype(np.float32)).unsqueeze(0),
                s.y[i], s.d[i], int(j))

# ---- طبقة عكس التدرّج ----
class GradReverse(Function):
    @staticmethod
    def forward(ctx,x,lam): ctx.lam=lam; return x.view_as(x)
    @staticmethod
    def backward(ctx,g): return g.neg()*ctx.lam, None
def grad_reverse(x,lam): return GradReverse.apply(x,lam)

# ---- النموذج: عمود فقري مشترك + رأس مرض + رأس قاعدة (خصم) ----
class DANN(nn.Module):
    def __init__(s,n_db):
        super().__init__()
        s.feat=nn.Sequential(
            nn.Conv2d(1,16,3,padding=1),nn.BatchNorm2d(16),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),nn.AdaptiveAvgPool2d(1))
        s.cls=nn.Linear(64,1)            # رأس المرض
        s.dom=nn.Sequential(nn.Linear(64,32),nn.ReLU(),nn.Linear(32,n_db))  # رأس القاعدة
    def forward(s,x,lam=0.0):
        f=s.feat(x).flatten(1)
        return s.cls(f).squeeze(1), s.dom(grad_reverse(f,lam))

def auc_rec(model,loader):
    model.eval(); P,Y,S=[],[],[]
    with torch.no_grad():
        for xb,yb,db,sid in loader:
            o,_=model(xb.to(device),0.0)
            P.append(torch.sigmoid(o).cpu().numpy()); Y.append(yb.numpy()); S.append(sid.numpy())
    P,Y,S=np.concatenate(P),np.concatenate(Y),np.concatenate(S)
    t=pd.DataFrame({"seg_id":S,"prob":P}).merge(seg_df[["seg_id","record","label"]],on="seg_id")
    rec=t.groupby("record").agg(prob=("prob","mean"),label=("label","first")).reset_index()
    y,p=rec["label"].values,rec["prob"].values
    if len(np.unique(y))<2: return np.nan,np.nan,np.nan
    yh=(p>=.5).astype(int)
    return roc_auc_score(y,p),balanced_accuracy_score(y,yh),f1_score(y,yh,zero_division=0)

def run(seed, adversarial):
    np.random.seed(seed); torch.manual_seed(seed)
    c="split_clean_database"
    tr=seg_df[seg_df[c]=="train"]; va=seg_df[seg_df[c]=="val"]; te=seg_df[seg_df[c]=="test"]
    trL=DataLoader(DS(tr),BATCH,shuffle=True,num_workers=2)
    vaL=DataLoader(DS(va),BATCH,shuffle=False,num_workers=2)
    teL=DataLoader(DS(te),BATCH,shuffle=False,num_workers=2)
    m=DANN(N_DB).to(device)
    pw=torch.tensor([(tr.label==0).sum()/max(1,(tr.label==1).sum())],device=device)
    crit_c=nn.BCEWithLogitsLoss(pos_weight=pw); crit_d=nn.CrossEntropyLoss()
    opt=torch.optim.Adam(m.parameters(),1e-3)
    best,bs,bad=-1,None,0
    for ep in range(EPOCHS):
        m.train()
        # lambda يتصاعد تدريجياً (قياسي في DANN)
        lam=(2.0/(1.0+np.exp(-10*ep/EPOCHS))-1.0) if adversarial else 0.0
        for xb,yb,db,_ in trL:
            xb,yb,db=xb.to(device),yb.to(device),db.to(device)
            opt.zero_grad()
            oc,od=m(xb,lam)
            loss=crit_c(oc,yb)+(crit_d(od,db) if adversarial else 0.0)
            loss.backward(); opt.step()
        a,_,_=auc_rec(m,vaL)
        if not np.isnan(a) and a>best: best,bs,bad=a,{k:v.cpu().clone() for k,v in m.state_dict().items()},0
        else:
            bad+=1
            if bad>=PATIENCE: break
    if bs: m.load_state_dict(bs)
    return auc_rec(m,teL)

print("\n[2] المقارنة: baseline (بلا خصم) مقابل DANN (مع خصم) ...\n")
rows=[]; t0=time.time()
for tag,adv in [("baseline",False),("DANN",True)]:
    for sd in SEEDS:
        t1=time.time(); auc,bal,f1=run(sd,adv); dt=time.time()-t1
        rows.append({"method":tag,"seed":sd,"auc":auc,"bal_acc":bal,"f1":f1})
        print(f"  {tag:9s} seed={sd:<4d} AUC={auc:.3f} bal={bal:.3f} f1={f1:.3f}  ({dt:.0f}s)")
        pd.DataFrame(rows).to_csv(os.path.join(OUT,"mitigation_results.csv"),index=False)

r=pd.DataFrame(rows)
print("\n"+"="*60+"\nالنتيجة: هل حسّن الخصم التعميم عبر القواعد؟\n"+"="*60)
g=r.groupby("method").agg(auc_m=("auc","mean"),auc_s=("auc","std"),
                           bal_m=("bal_acc","mean"),f1_m=("f1","mean"))
for mth in ["baseline","DANN"]:
    if mth in g.index:
        x=g.loc[mth]; print(f"  {mth:9s}: AUC={x.auc_m:.3f}±{x.auc_s:.3f}  bal={x.bal_m:.3f}  F1={x.f1_m:.3f}")

from scipy import stats
xb=r[r.method=="baseline"].sort_values("seed")["auc"].values
xd=r[r.method=="DANN"].sort_values("seed")["auc"].values
print(f"\n  تحسّن AUC = {xd.mean()-xb.mean():+.3f}")
print(f"  paired t-test p = {stats.ttest_rel(xd,xb).pvalue:.4f}")
print("""
التفسير:
- لو AUC(DANN) > AUC(baseline) بوضوح: الحل ينجح، لدينا "كشف+قياس+حل" ✅
- لو التحسّن ضئيل: نجرّب بدائل (موازنة المصادر / تطبيع لكل قاعدة)
""")
print("✅ حُفظ في results/mitigation_results.csv")

الجهاز: cuda

[1] تحميل البيانات ...
    القواعد: ['a', 'b', 'c', 'd', 'e', 'f'] (عددها 6)

[2] المقارنة: baseline (بلا خصم) مقابل DANN (مع خصم) ...

  baseline  seed=42   AUC=0.437 bal=0.433 f1=0.512  (47s)
  baseline  seed=123  AUC=0.327 bal=0.484 f1=0.583  (42s)
  baseline  seed=2024 AUC=0.356 bal=0.467 f1=0.560  (47s)
  baseline  seed=7    AUC=0.341 bal=0.487 f1=0.589  (46s)
  baseline  seed=99   AUC=0.612 bal=0.507 f1=0.603  (47s)
  DANN      seed=42   AUC=0.488 bal=0.435 f1=0.496  (47s)
  DANN      seed=123  AUC=0.578 bal=0.500 f1=0.601  (33s)
  DANN      seed=2024 AUC=0.387 bal=0.417 f1=0.498  (37s)
  DANN      seed=7    AUC=0.375 bal=0.368 f1=0.426  (47s)
  DANN      seed=99   AUC=0.527 bal=0.481 f1=0.577  (47s)

النتيجة: هل حسّن الخصم التعميم عبر القواعد؟
  baseline : AUC=0.414±0.118  bal=0.476  F1=0.569
  DANN     : AUC=0.471±0.088  bal=0.440  F1=0.520

  تحسّن AUC = +0.057
  paired t-test p = 0.3552

التفسير:
- لو AUC(DANN) > AUC(baseline) بوضوح: الحل ينجح، لدينا "كشف+قياس+ح

In [17]:
# =============================================================
#  Generate paper figures from saved results
#  No training - plotting only. Fast.
#  Saves each figure as PDF and PNG (300 dpi) for publication
# =============================================================

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

PREP = "/kaggle/working/prepared"
OUT  = "/kaggle/working/figures"
os.makedirs(OUT, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300,
    "font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "savefig.bbox": "tight"
})

def save(fig, name):
    for ext in ["pdf", "png"]:
        fig.savefig(os.path.join(OUT, f"{name}.{ext}"))
    print(f"   saved: {name}.pdf / .png")

# -------------------------------------------------------------
# Load results
# -------------------------------------------------------------
res = pd.read_csv("/kaggle/working/results/final_results.csv")
rec = res[res["eval"] == "record"]

PROTOS = ["leaky_segment", "clean_record", "clean_database"]
LABELS = ["Segment leakage\n(common, flawed)",
          "Record-level\n(assumed clean)",
          "Cross-database\n(true generalization)"]
COLORS = ["#A32D2D", "#BA7517", "#0F6E56"]

means = [rec[rec.protocol == p]["auc"].mean() for p in PROTOS]
sds   = [rec[rec.protocol == p]["auc"].std()  for p in PROTOS]

# =============================================================
# Figure 1: AUC collapse across protocols (main figure)
# =============================================================
print("\n[1] AUC collapse bar chart ...")
fig, ax = plt.subplots(figsize=(7, 4.5))
x = np.arange(3)
ax.bar(x, means, yerr=sds, capsize=6, color=COLORS,
       edgecolor="black", linewidth=0.6, width=0.6,
       error_kw={"elinewidth": 1.3})
ax.axhline(0.5, ls="--", color="gray", lw=1, label="Random guess (0.5)")
ax.set_xticks(x); ax.set_xticklabels(LABELS)
ax.set_ylabel("AUC (record-level)"); ax.set_ylim(0, 1.0)
ax.set_title("Performance collapse across evaluation protocols")
for i, (m, s) in enumerate(zip(means, sds)):
    ax.text(i, m + s + 0.02, f"{m:.3f}", ha="center", fontsize=10, fontweight="bold")
ax.legend(frameon=False, loc="upper right")
save(fig, "fig1_auc_collapse"); plt.close(fig)

# =============================================================
# Figure 2: class distribution heatmap across databases
# =============================================================
print("[2] class distribution heatmap ...")
seg = pd.read_csv(os.path.join(PREP, "seg_index_with_splits.csv"))
recs = seg.drop_duplicates("record")[["record", "db", "label"]]
tab = pd.crosstab(recs["db"], recs["label"])
tab.columns = ["healthy", "unhealthy"]

fig, ax = plt.subplots(figsize=(6, 4))
mat = tab.values.astype(float)
im = ax.imshow(mat, cmap="YlOrRd", aspect="auto")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Healthy", "Unhealthy"])
ax.set_yticks(range(len(tab.index)))
ax.set_yticklabels([f"DB {d}" for d in tab.index])
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        ax.text(j, i, int(mat[i, j]), ha="center", va="center",
                color="black" if mat[i, j] < mat.max() * 0.6 else "white",
                fontsize=10)
ax.set_title("Class distribution per source database")
fig.colorbar(im, ax=ax, label="Number of recordings")
save(fig, "fig2_class_heatmap"); plt.close(fig)

# =============================================================
# Figure 3: performance variability boxplot across seeds
# =============================================================
print("[3] variability boxplot ...")
fig, ax = plt.subplots(figsize=(7, 4.5))
data = [rec[rec.protocol == p]["auc"].values for p in PROTOS]
bp = ax.boxplot(data, patch_artist=True, widths=0.5,
                medianprops={"color": "black", "linewidth": 1.3})
for patch, c in zip(bp["boxes"], COLORS):
    patch.set_facecolor(c); patch.set_alpha(0.75); patch.set_edgecolor("black")
ax.axhline(0.5, ls="--", color="gray", lw=1)
ax.set_xticklabels(LABELS)
ax.set_ylabel("AUC (record-level)"); ax.set_ylim(0, 1.0)
ax.set_title("Performance variability across 5 seeds")
save(fig, "fig3_variability_boxplot"); plt.close(fig)

print("\nDone. All figures in /kaggle/working/figures/")
print("To download: use the Output panel on the right.")


[1] AUC collapse bar chart ...
   saved: fig1_auc_collapse.pdf / .png
[2] class distribution heatmap ...
   saved: fig2_class_heatmap.pdf / .png
[3] variability boxplot ...
   saved: fig3_variability_boxplot.pdf / .png

Done. All figures in /kaggle/working/figures/
To download: use the Output panel on the right.


In [18]:
import os, glob, subprocess

# 1) هل الإنترنت مفعّل؟
import urllib.request
try:
    urllib.request.urlopen("https://www.kaggle.com", timeout=10)
    print("✓ الإنترنت مفعّل")
except Exception as e:
    print("✗ الإنترنت معطّل — فعّليه: Settings → Internet → On")
    print("  ", e)

# 2) هل بيانات اعتماد kaggle مهيّأة؟
print("kaggle.json موجود؟", os.path.exists("/root/.kaggle/kaggle.json") 
      or os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")))

# 3) جرّبي التحميل فعلياً
print("\nجاري التحميل...")
r = subprocess.run(
    "kaggle datasets download -d swapnilpanda/heart-sound-database -p /kaggle/working --unzip",
    shell=True, capture_output=True, text=True)
print("stdout:", r.stdout[-500:])
print("stderr:", r.stderr[-500:])

# 4) تحقّقي من النتيجة
wavs = glob.glob("/kaggle/working/**/*.wav", recursive=True)
print("\nملفات wav بعد التحميل:", len(wavs))
if wavs:
    print("أمثلة:", wavs[:3])

✓ الإنترنت مفعّل
kaggle.json موجود؟ False

جاري التحميل...
stdout: Warning: Looks like you're using an outdated `kaggle` version (installed: 2.0.0), please consider upgrading to the latest version (2.0.2)
Dataset URL: https://www.kaggle.com/datasets/swapnilpanda/heart-sound-database
License(s): unknown


stderr: | 49.0M/173M [00:00<00:01, 65.4MB/s]
 34%|███▍      | 59.0M/173M [00:00<00:01, 67.6MB/s]
 42%|████▏     | 73.0M/173M [00:01<00:01, 84.0MB/s]
 48%|████▊     | 83.0M/173M [00:01<00:01, 81.3MB/s]
 55%|█████▍    | 95.0M/173M [00:01<00:00, 91.3MB/s]
 61%|██████    | 105M/173M [00:01<00:00, 94.4MB/s] 
 70%|██████▉   | 121M/173M [00:01<00:00, 81.7MB/s]
 84%|████████▎ | 145M/173M [00:01<00:00, 114MB/s] 
 92%|█████████▏| 159M/173M [00:01<00:00, 110MB/s]
100%|██████████| 173M/173M [00:01<00:00, 94.3MB/s]


ملفات wav بعد التحميل: 3541
أمثلة: ['/kaggle/working/heart_sound/train/healthy/e01908.wav', '/kaggle/working/heart_sound/train/healthy/e01439.wav', '/kaggle/working/heart_sound/train/h

In [19]:
# ============================================================
# خلية واحدة شاملة: بناء البيانات + التجارب الباقية (سريعة)
# ============================================================
import os, glob, wave, itertools
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, confusion_matrix
from sklearn.manifold import TSNE

class CFG:
    raw_root="/kaggle/working/heart_sound"; prep_dir="/kaggle/working"
    sr=2000; win_sec=3.0; hop_sec=1.5; n_mels=40; n_fft=256; hop_fft=64
CFG.win=int(CFG.sr*CFG.win_sec); CFG.hop=int(CFG.sr*CFG.hop_sec); OUT=CFG.prep_dir

def read_wav(p):
    with wave.open(p,"rb") as wf:
        x=np.frombuffer(wf.readframes(wf.getnframes()),dtype=np.int16).astype(np.float32)
    return (x-x.mean())/(x.std()+1e-6) if x.std()>1e-6 else x

def build_all():
    wavs=glob.glob(os.path.join(CFG.raw_root,"**","*.wav"),recursive=True)
    print("[1] wav files:",len(wavs))
    rows={}
    for w in wavs:
        rec=os.path.splitext(os.path.basename(w))[0]; parts=w.lower().split(os.sep)
        if "unhealthy" in parts: lab=1
        elif "healthy" in parts: lab=0
        else: continue
        d=rec[0] if rec[:1].isalpha() else "?"
        rows.setdefault(rec,{"record":rec,"path":w,"db":d,"label":lab})
    man=pd.DataFrame(rows.values()).sort_values("record").reset_index(drop=True)
    print(f"[2] records: {len(man)} (healthy={(man.label==0).sum()}, unhealthy={(man.label==1).sum()})")
    segs,idx=[],[]
    for _,r in man.iterrows():
        x=read_wav(r["path"])
        if len(x)<CFG.win: x=np.pad(x,(0,CFG.win-len(x)))
        starts=list(range(0,len(x)-CFG.win+1,CFG.hop)) or [0]
        for s in starts:
            idx.append({"record":r["record"],"db":r["db"],"label":int(r["label"])})
            segs.append(x[s:s+CFG.win].astype(np.float16))
    seg_df=pd.DataFrame(idx); segs=np.stack(segs)
    print(f"[3] segments: {len(seg_df)} (avg {len(seg_df)/len(man):.1f}/rec)")
    cache=os.path.join(CFG.prep_dir,"mels.npy")
    if os.path.exists(cache):
        print("[4] loading cached mels"); M=np.load(cache)
    else:
        import torch, torchaudio
        mel=torchaudio.transforms.MelSpectrogram(sample_rate=CFG.sr,n_fft=CFG.n_fft,
            hop_length=CFG.hop_fft,n_mels=CFG.n_mels,power=2.0)
        todb=torchaudio.transforms.AmplitudeToDB(stype="power"); out=[]
        for i in range(0,len(segs),512):
            m=todb(mel(torch.from_numpy(segs[i:i+512].astype(np.float32))))
            m=(m-m.mean(dim=(1,2),keepdim=True))/(m.std(dim=(1,2),keepdim=True)+1e-6)
            out.append(m.numpy().astype(np.float16))
        M=np.concatenate(out); np.save(cache,M); print(f"[4] mel shape: {M.shape}")
    return M.astype(np.float32), seg_df["label"].values.astype(int), seg_df["record"].values, seg_df["db"].values

# ---- بناء ----
X,y,rec,db = build_all()
print(f"\n✓ جاهز: {len(y)} segments، {len(set(db))} قواعد {sorted(set(db))}، شكل X: {X.shape}\n")

# ---- أدوات ----
def mel_to_vector(X): return np.concatenate([X.mean(axis=2),X.std(axis=2)],axis=1)
def to_record(p,ys,r):
    df=pd.DataFrame({"p":p,"y":ys,"r":r}); g=df.groupby("r").agg(p=("p","mean"),y=("y","first"))
    return g["p"].values,g["y"].values
feats_all=mel_to_vector(X)

def split_idx(proto,seed=42):
    rng=np.random.default_rng(seed); n=len(y); idx=np.arange(n)
    if proto=="leaky_segment": rng.shuffle(idx); return idx[:int(0.7*n)],idx[int(0.7*n):]
    if proto=="clean_record":
        recs=np.array(sorted(set(rec))); rng.shuffle(recs)
        ts=set(recs[:int(0.7*len(recs))]); tr=np.array([r in ts for r in rec]); return idx[tr],idx[~tr]
    if proto=="clean_database":
        dbs=sorted(set(db)); rng.shuffle(dbs)
        ts=set(dbs[:max(1,int(0.7*len(dbs)))]); tr=np.array([d in ts for d in db]); return idx[tr],idx[~tr]

# ---- (3) source probe أولاً (الأهم والأسرع) ----
def fast_source_probe(max_n=8000):
    rng=np.random.default_rng(0)
    i=rng.choice(len(feats_all),min(max_n,len(feats_all)),replace=False)
    F=feats_all[i]; D=np.array(db)[i]; rows=[]
    for t in sorted(set(D)):
        z=(D==t).astype(int)
        if z.sum()<10: continue
        ix=np.arange(len(z)); rng.shuffle(ix); c=int(0.7*len(ix)); tr,te=ix[:c],ix[c:]
        sc=StandardScaler().fit(F[tr])
        clf=RandomForestClassifier(n_estimators=150,n_jobs=-1).fit(sc.transform(F[tr]),z[tr])
        p=clf.predict_proba(sc.transform(F[te]))[:,1]
        rows.append(dict(database=t,source_auc=roc_auc_score(z[te],p)))
    df=pd.DataFrame(rows); df.to_csv(f"{OUT}/source_probe.csv",index=False)
    print("[SOURCE PROBE]\n",df.round(3))
    print(f"[SOURCE PROBE] mean = {df.source_auc.mean():.3f} (>>0.5 = source decodable)"); return df

# ---- (2) baselines سريعة ----
def fast_baselines(seeds=(42,123,2024,7,99)):
    rows=[]; models={"LogReg":lambda:LogisticRegression(max_iter=1000,n_jobs=-1),
                     "LinearSVM":lambda:CalibratedClassifierCV(LinearSVC(max_iter=2000),cv=3)}
    for proto in ["leaky_segment","clean_record","clean_database"]:
        for name,fac in models.items():
            for s in seeds:
                tr,te=split_idx(proto,seed=s)
                if len(set(y[tr]))<2 or len(set(y[te]))<2: continue
                sc=StandardScaler().fit(feats_all[tr])
                clf=fac().fit(sc.transform(feats_all[tr]),y[tr])
                p=clf.predict_proba(sc.transform(feats_all[te]))[:,1]
                pr,yr=to_record(p,y[te],rec[te])
                if len(set(yr))<2: continue
                rows.append(dict(protocol=proto,model=name,seed=s,auc=roc_auc_score(yr,pr),
                    bal_acc=balanced_accuracy_score(yr,(pr>0.5).astype(int))))
    df=pd.DataFrame(rows); df.to_csv(f"{OUT}/baseline_results.csv",index=False)
    print("\n[BASELINE] mean AUC by protocol x model:")
    print(df.groupby(["protocol","model"]).auc.agg(["mean","std"]).round(3)); return df

# ---- (4) confusion + t-SNE ----
def fast_confusion(seed=42):
    fig,axes=plt.subplots(1,3,figsize=(13,4))
    for ax,proto in zip(axes,["leaky_segment","clean_record","clean_database"]):
        tr,te=split_idx(proto,seed=seed)
        sc=StandardScaler().fit(feats_all[tr])
        clf=RandomForestClassifier(n_estimators=150,n_jobs=-1).fit(sc.transform(feats_all[tr]),y[tr])
        p=clf.predict_proba(sc.transform(feats_all[te]))[:,1]; pr,yr=to_record(p,y[te],rec[te])
        cm=confusion_matrix(yr,(pr>0.5).astype(int)); ax.imshow(cm,cmap="Blues"); ax.set_title(proto)
        ax.set_xticks([0,1]); ax.set_xticklabels(["normal","abnormal"])
        ax.set_yticks([0,1]); ax.set_yticklabels(["normal","abnormal"])
        for a in range(2):
            for b in range(2): ax.text(b,a,cm[a,b],ha="center",va="center")
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    plt.tight_layout(); fig.savefig(f"{OUT}/fig_confusion_matrices.png",dpi=300); print("saved fig_confusion_matrices.png")

def fast_tsne(max_pts=3000):
    rng=np.random.default_rng(0); i=rng.choice(len(feats_all),min(max_pts,len(feats_all)),replace=False)
    F=StandardScaler().fit_transform(feats_all[i]); y2=y[i]; db2=np.array(db)[i]
    emb=TSNE(n_components=2,init="pca",perplexity=30,random_state=0).fit_transform(F)
    fig,ax=plt.subplots(1,2,figsize=(12,5))
    for d in sorted(set(db2)):
        m=db2==d; ax[0].scatter(emb[m,0],emb[m,1],s=4,label=f"DB {d}",alpha=0.5)
    ax[0].legend(markerscale=3,fontsize=8); ax[0].set_title("t-SNE by SOURCE DATABASE")
    for c,lab in [(0,"normal"),(1,"abnormal")]:
        m=y2==c; ax[1].scatter(emb[m,0],emb[m,1],s=4,label=lab,alpha=0.5)
    ax[1].legend(markerscale=3,fontsize=8); ax[1].set_title("t-SNE by CLASS")
    plt.tight_layout(); fig.savefig(f"{OUT}/fig_tsne_embeddings.png",dpi=300); print("saved fig_tsne_embeddings.png")

# ---- شغّلي ----
fast_source_probe()
fast_baselines()
fast_confusion()
fast_tsne()
print("\n✓ انتهت كل التجارب الباقية")

[1] wav files: 3541
[2] records: 3240 (healthy=2575, unhealthy=665)
[3] segments: 43678 (avg 13.5/rec)
[4] mel shape: (43678, 40, 94)

✓ جاهز: 43678 segments، 6 قواعد ['a', 'b', 'c', 'd', 'e', 'f']، شكل X: (43678, 40, 94)

[SOURCE PROBE]
   database  source_auc
0        a       0.995
1        b       0.997
2        c       0.997
3        d       0.993
4        e       0.999
5        f       1.000
[SOURCE PROBE] mean = 0.997 (>>0.5 = source decodable)

[BASELINE] mean AUC by protocol x model:
                           mean    std
protocol       model                  
clean_database LinearSVM  0.455  0.137
               LogReg     0.457  0.132
clean_record   LinearSVM  0.926  0.010
               LogReg     0.929  0.011
leaky_segment  LinearSVM  0.941  0.002
               LogReg     0.940  0.002
saved fig_confusion_matrices.png
saved fig_tsne_embeddings.png

✓ انتهت كل التجارب الباقية


In [ ]:
import itertools
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, f1_score, balanced_accuracy_score, confusion_matrix)
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt

def mel_to_vector(X): return np.concatenate([X.mean(axis=2), X.std(axis=2)], axis=1)

def to_record(probs, y_seg, rec_id):
    df = pd.DataFrame({"p": probs, "y": y_seg, "r": rec_id})
    g = df.groupby("r").agg(p=("p","mean"), y=("y","first"))
    return g["p"].values, g["y"].values

def run_lodo(X,y,rec,db):
    feats=mel_to_vector(X); dbs=sorted(set(db)); rows=[]
    for held in dbs:
        tr,te=db!=held,db==held
        if len(set(y[tr]))<2 or len(set(y[te]))<2:
            rows.append(dict(held_out=held,auc=np.nan,note="single-class")); continue
        sc=StandardScaler().fit(feats[tr])
        clf=RandomForestClassifier(n_estimators=200,n_jobs=-1).fit(sc.transform(feats[tr]),y[tr])
        p=clf.predict_proba(sc.transform(feats[te]))[:,1]
        pr,yr=to_record(p,y[te],rec[te])
        rows.append(dict(held_out=held,auc=roc_auc_score(yr,pr) if len(set(yr))>1 else np.nan,
            f1=f1_score(yr,(pr>0.5).astype(int)),
            bal_acc=balanced_accuracy_score(yr,(pr>0.5).astype(int)),n_test=int(te.sum())))
    df=pd.DataFrame(rows); df.to_csv(f"{OUT}/lodo_results.csv",index=False)
    print("\n[LODO]\n",df); print(f"[LODO] mean AUC = {df.auc.mean():.3f} ± {df.auc.std():.3f}")
    return df

def run_matrix(X,y,rec,db):
    feats=mel_to_vector(X); dbs=sorted(set(db)); M=pd.DataFrame(index=dbs,columns=dbs,dtype=float)
    for a,b in itertools.product(dbs,dbs):
        tr,te=db==a,db==b
        if len(set(y[tr]))<2 or len(set(y[te]))<2: continue
        sc=StandardScaler().fit(feats[tr])
        clf=RandomForestClassifier(n_estimators=200,n_jobs=-1).fit(sc.transform(feats[tr]),y[tr])
        p=clf.predict_proba(sc.transform(feats[te]))[:,1]
        pr,yr=to_record(p,y[te],rec[te])
        if len(set(yr))>1: M.loc[a,b]=roc_auc_score(yr,pr)
    M.to_csv(f"{OUT}/generalization_matrix.csv"); print("\n[MATRIX]\n",M.round(3))
    fig,ax=plt.subplots(figsize=(6,5))
    im=ax.imshow(M.values.astype(float),vmin=0,vmax=1,cmap="RdYlGn")
    ax.set_xticks(range(len(dbs))); ax.set_xticklabels(dbs)
    ax.set_yticks(range(len(dbs))); ax.set_yticklabels(dbs)
    ax.set_xlabel("Test database"); ax.set_ylabel("Train database")
    ax.set_title("Cross-database generalization (record-level AUC)")
    for i in range(len(dbs)):
        for j in range(len(dbs)):
            v=M.values[i,j]
            if not np.isnan(v): ax.text(j,i,f"{v:.2f}",ha="center",va="center",fontsize=8)
    plt.colorbar(im); plt.tight_layout(); fig.savefig(f"{OUT}/fig_generalization_matrix.png",dpi=300)
    print("saved fig_generalization_matrix.png"); return M

def split_idx(protocol,y,rec,db,seed=42):
    rng=np.random.default_rng(seed); n=len(y); idx=np.arange(n)
    if protocol=="leaky_segment":
        rng.shuffle(idx); return idx[:int(0.7*n)], idx[int(0.7*n):]
    if protocol=="clean_record":
        recs=np.array(sorted(set(rec))); rng.shuffle(recs)
        trset=set(recs[:int(0.7*len(recs))]); tr=np.array([r in trset for r in rec])
        return idx[tr], idx[~tr]
    if protocol=="clean_database":
        dbs=sorted(set(db)); rng.shuffle(dbs)
        trset=set(dbs[:max(1,int(0.7*len(dbs)))]); tr=np.array([d in trset for d in db])
        return idx[tr], idx[~tr]

def run_baselines(X,y,rec,db,seeds=(42,123,2024,7,99)):
    feats=mel_to_vector(X); rows=[]
    models={"SVM":lambda:SVC(probability=True,kernel="rbf",C=1.0),
            "RandomForest":lambda:RandomForestClassifier(n_estimators=200,n_jobs=-1)}
    for proto in ["leaky_segment","clean_record","clean_database"]:
        for name,fac in models.items():
            for s in seeds:
                tr,te=split_idx(proto,y,rec,db,seed=s)
                if len(set(y[tr]))<2 or len(set(y[te]))<2: continue
                sc=StandardScaler().fit(feats[tr])
                clf=fac().fit(sc.transform(feats[tr]),y[tr])
                p=clf.predict_proba(sc.transform(feats[te]))[:,1]
                pr,yr=to_record(p,y[te],rec[te])
                if len(set(yr))<2: continue
                rows.append(dict(protocol=proto,model=name,seed=s,auc=roc_auc_score(yr,pr),
                    bal_acc=balanced_accuracy_score(yr,(pr>0.5).astype(int))))
    df=pd.DataFrame(rows); df.to_csv(f"{OUT}/baseline_results.csv",index=False)
    print("\n[BASELINE] mean AUC by protocol x model:")
    print(df.groupby(["protocol","model"]).auc.agg(["mean","std"]).round(3)); return df

def run_source_probe(X,y,db):
    feats=mel_to_vector(X); dbs=sorted(set(db)); rows=[]; rng=np.random.default_rng(0)
    for target in dbs:
        z=(db==target).astype(int)
        if z.sum()<10: continue
        idx=np.arange(len(z)); rng.shuffle(idx); cut=int(0.7*len(idx))
        tr,te=idx[:cut],idx[cut:]
        sc=StandardScaler().fit(feats[tr])
        clf=RandomForestClassifier(n_estimators=200,n_jobs=-1).fit(sc.transform(feats[tr]),z[tr])
        p=clf.predict_proba(sc.transform(feats[te]))[:,1]
        rows.append(dict(database=target,source_auc=roc_auc_score(z[te],p)))
    df=pd.DataFrame(rows); df.to_csv(f"{OUT}/source_probe.csv",index=False)
    print("\n[SOURCE PROBE]\n",df.round(3))
    print(f"[SOURCE PROBE] mean = {df.source_auc.mean():.3f} (>>0.5 = source decodable)"); return df

def run_tsne(X,y,db,max_pts=3000):
    feats=mel_to_vector(X)
    if len(feats)>max_pts:
        i=np.random.default_rng(0).choice(len(feats),max_pts,replace=False)
        feats,y2,db2=feats[i],y[i],np.array(db)[i]
    else: y2,db2=y,np.array(db)
    emb=TSNE(n_components=2,init="pca",perplexity=30,random_state=0).fit_transform(
        StandardScaler().fit_transform(feats))
    fig,ax=plt.subplots(1,2,figsize=(12,5))
    for d in sorted(set(db2)):
        m=db2==d; ax[0].scatter(emb[m,0],emb[m,1],s=4,label=f"DB {d}",alpha=0.5)
    ax[0].legend(markerscale=3,fontsize=8); ax[0].set_title("t-SNE by SOURCE DATABASE")
    for c,lab in [(0,"normal"),(1,"abnormal")]:
        m=y2==c; ax[1].scatter(emb[m,0],emb[m,1],s=4,label=lab,alpha=0.5)
    ax[1].legend(markerscale=3,fontsize=8); ax[1].set_title("t-SNE by CLASS")
    plt.tight_layout(); fig.savefig(f"{OUT}/fig_tsne_embeddings.png",dpi=300)
    print("saved fig_tsne_embeddings.png")

def run_confusion(X,y,rec,db,seed=42):
    feats=mel_to_vector(X); fig,axes=plt.subplots(1,3,figsize=(13,4))
    for ax,proto in zip(axes,["leaky_segment","clean_record","clean_database"]):
        tr,te=split_idx(proto,y,rec,db,seed=seed)
        sc=StandardScaler().fit(feats[tr])
        clf=RandomForestClassifier(n_estimators=200,n_jobs=-1).fit(sc.transform(feats[tr]),y[tr])
        p=clf.predict_proba(sc.transform(feats[te]))[:,1]
        pr,yr=to_record(p,y[te],rec[te])
        cm=confusion_matrix(yr,(pr>0.5).astype(int))
        ax.imshow(cm,cmap="Blues"); ax.set_title(proto)
        ax.set_xticks([0,1]); ax.set_xticklabels(["normal","abnormal"])
        ax.set_yticks([0,1]); ax.set_yticklabels(["normal","abnormal"])
        for i in range(2):
            for j in range(2): ax.text(j,i,cm[i,j],ha="center",va="center")
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    plt.tight_layout(); fig.savefig(f"{OUT}/fig_confusion_matrices.png",dpi=300)
    print("saved fig_confusion_matrices.png")

# ===== شغّلي التجارب الستّ =====
run_lodo(X,y,rec,db)
run_matrix(X,y,rec,db)
run_baselines(X,y,rec,db)
run_source_probe(X,y,db)
run_tsne(X,y,db)
run_confusion(X,y,rec,db)
print("\n✓ انتهت كل التجارب — ارفعي ملفات CSV والصور")

In [21]:
# source probe فقط — سريع
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

try:
    feats_all
except NameError:
    feats_all = np.concatenate([X.mean(axis=2), X.std(axis=2)], axis=1)

rng=np.random.default_rng(0)
i=rng.choice(len(feats_all), min(8000,len(feats_all)), replace=False)
F=feats_all[i]; D=np.array(db)[i]; rows=[]
for t in sorted(set(D)):
    z=(D==t).astype(int)
    if z.sum()<10: continue
    ix=np.arange(len(z)); rng.shuffle(ix); c=int(0.7*len(ix)); tr,te=ix[:c],ix[c:]
    sc=StandardScaler().fit(F[tr])
    clf=RandomForestClassifier(n_estimators=150,n_jobs=-1).fit(sc.transform(F[tr]),z[tr])
    p=clf.predict_proba(sc.transform(F[te]))[:,1]
    rows.append((t, roc_auc_score(z[te],p)))
import pandas as pd
df=pd.DataFrame(rows,columns=["database","source_auc"])
df.to_csv("/kaggle/working/source_probe.csv",index=False)
print(df.round(3))
print(f"\n[SOURCE PROBE] mean = {df.source_auc.mean():.3f}  (>>0.5 = source strongly decodable)")

  database  source_auc
0        a       0.996
1        b       0.997
2        c       0.996
3        d       0.992
4        e       0.999
5        f       1.000

[SOURCE PROBE] mean = 0.997  (>>0.5 = source strongly decodable)


In [ ]:
"""
================================================================================
TWO EXPERIMENTS IN ONE  (Reviewer concerns #1 single-dataset, #2 no mitigation)
================================================================================
EXP A — EXTERNAL VALIDATION on PASCAL:
   Train on ALL of PhysioNet, test on PASCAL (a fully independent corpus,
   different devices/protocol). If performance collapses, the leakage/transfer
   problem is NOT specific to PhysioNet -> the paper becomes multi-dataset.

EXP B — MITIGATION:
   Test whether a simple, CPU-friendly domain-mitigation step recovers any
   cross-database performance:
     (B1) per-database feature standardization (remove each source's mean/var)
     (B2) "leakage-aware" training: standardize features within each source
          before LODO, so the model cannot exploit raw source offsets.
   If LODO AUC improves above ~0.5, mitigation is partially effective and the
   paper becomes constructive, not merely diagnostic.

SETUP
-----
You need TWO datasets attached on Kaggle:
  1) PhysioNet (already used): swapnilpanda/heart-sound-database
  2) PASCAL heart sounds. Common Kaggle slug: kinguistics/heartbeat-sounds
     (folders set_a / set_b with files named like 'normal__*', 'murmur__*',
      'extrastole__*', 'artifact__*', 'extrahls__*').
If your PASCAL slug differs, set PASCAL_ROOT to its path under /kaggle/input.

This reuses your PhysioNet build_all(). Run that first (so X,y,rec,db exist),
or this script will rebuild PhysioNet too (it imports the same helpers).
================================================================================
"""
import os, glob, wave, numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
OUT = "/kaggle/working"

# ---------------------------------------------------------------- config
SR = 2000; WIN = int(SR*3.0); HOP = int(SR*1.5); N_MELS=40; N_FFT=256; HOP_FFT=64
PHYSIO_ROOT = "/kaggle/working/heart_sound"
# Try to auto-find PASCAL under /kaggle/input
def find_pascal():
    cands = glob.glob("/kaggle/input/**/set_a", recursive=True) + \
            glob.glob("/kaggle/input/**/set_b", recursive=True) + \
            glob.glob("/kaggle/input/**/*ascal*", recursive=True)
    if cands:
        # return the parent that contains set_a/set_b or wavs
        root = os.path.dirname(cands[0]) if cands[0].endswith(("set_a","set_b")) else cands[0]
        return root
    return None
PASCAL_ROOT = find_pascal()
print("PASCAL_ROOT =", PASCAL_ROOT)

# ---------------------------------------------------------------- audio + mel
def read_wav(p):
    try:
        with wave.open(p,"rb") as wf:
            sr = wf.getframerate()
            x = np.frombuffer(wf.readframes(wf.getnframes()), dtype=np.int16).astype(np.float32)
    except Exception:
        return None, None
    return x, sr

def resample_to(x, sr_in, sr_out=SR):
    if sr_in == sr_out or sr_in is None: return x
    n = int(round(len(x) * sr_out / sr_in))
    if n < 1: return x
    return np.interp(np.linspace(0, len(x)-1, n), np.arange(len(x)), x)

def norm(x): return (x - x.mean())/(x.std()+1e-6) if x.std()>1e-6 else x

def mel_batch(segs):
    import torch, torchaudio
    mel = torchaudio.transforms.MelSpectrogram(sample_rate=SR, n_fft=N_FFT,
            hop_length=HOP_FFT, n_mels=N_MELS, power=2.0)
    todb = torchaudio.transforms.AmplitudeToDB(stype="power")
    out=[]
    for i in range(0,len(segs),512):
        m=todb(mel(torch.from_numpy(segs[i:i+512].astype(np.float32))))
        m=(m-m.mean(dim=(1,2),keepdim=True))/(m.std(dim=(1,2),keepdim=True)+1e-6)
        out.append(m.numpy().astype(np.float16))
    return np.concatenate(out)

def feats_from_mel(M): return np.concatenate([M.mean(axis=2), M.std(axis=2)], axis=1)

# ---------------------------------------------------------------- PASCAL loader
def load_pascal(root):
    if root is None:
        print("!! PASCAL not found. Attach a PASCAL dataset and set PASCAL_ROOT."); return None
    wavs = glob.glob(os.path.join(root,"**","*.wav"), recursive=True) + \
           glob.glob(os.path.join(root,"**","*.aiff"), recursive=True)
    print(f"PASCAL wavs found: {len(wavs)}")
    segs, labs = [], []
    for w in wavs:
        name = os.path.basename(w).lower()
        if name.startswith("artifact") or "artifact" in name:   # drop non-cardiac
            continue
        if name.startswith("normal") or "normal" in name: lab = 0
        elif any(k in name for k in ["murmur","extrastole","extrahls","extra"]): lab = 1
        else: continue
        x, sr = read_wav(w)
        if x is None or len(x) < 10: continue
        x = norm(resample_to(x, sr))
        if len(x) < WIN: x = np.pad(x,(0,WIN-len(x)))
        for s in range(0, len(x)-WIN+1, HOP) or [0]:
            segs.append(x[s:s+WIN].astype(np.float16)); labs.append(lab)
    if not segs:
        print("!! no PASCAL segments parsed"); return None
    segs=np.stack(segs); labs=np.array(labs)
    print(f"PASCAL segments: {len(labs)} (normal={ (labs==0).sum() }, abnormal={ (labs==1).sum() })")
    M = mel_batch(segs)
    return feats_from_mel(M.astype(np.float32)), labs

# ---------------------------------------------------------------- EXP A
def exp_a_external(Xtr_feat, ytr, pascal):
    if pascal is None: return
    Xpa, ypa = pascal
    sc = StandardScaler().fit(Xtr_feat)
    clf = RandomForestClassifier(n_estimators=200, n_jobs=-1).fit(sc.transform(Xtr_feat), ytr)
    p = clf.predict_proba(sc.transform(Xpa))[:,1]
    auc = roc_auc_score(ypa, p); bal = balanced_accuracy_score(ypa,(p>0.5).astype(int))
    pd.DataFrame([dict(train="PhysioNet(all)", test="PASCAL", auc=auc, bal_acc=bal,
                       n_test=len(ypa))]).to_csv(f"{OUT}/external_pascal.csv", index=False)
    print(f"\n[EXP A] Train PhysioNet -> Test PASCAL: AUC={auc:.3f}, bal_acc={bal:.3f}, n={len(ypa)}")
    print("  (collapse here => the transfer failure is NOT specific to PhysioNet)")

# ---------------------------------------------------------------- EXP B
def to_record(p, ys, r):
    df=pd.DataFrame({"p":p,"y":ys,"r":r}); g=df.groupby("r").agg(p=("p","mean"),y=("y","first"))
    return g["p"].values, g["y"].values

def lodo_with_optional_mitigation(X, y, rec, db, mitigate=False):
    feats = np.concatenate([X.mean(axis=2), X.std(axis=2)], axis=1)
    if mitigate:
        # per-database standardization: remove each source's own mean/std
        feats = feats.copy()
        for d in sorted(set(db)):
            m = db==d
            mu, sd = feats[m].mean(0), feats[m].std(0)+1e-6
            feats[m] = (feats[m]-mu)/sd
    aucs=[]
    for held in sorted(set(db)):
        tr,te = db!=held, db==held
        if len(set(y[tr]))<2 or len(set(y[te]))<2: continue
        sc=StandardScaler().fit(feats[tr])
        clf=RandomForestClassifier(n_estimators=150,n_jobs=-1).fit(sc.transform(feats[tr]),y[tr])
        p=clf.predict_proba(sc.transform(feats[te]))[:,1]
        pr,yr=to_record(p,y[te],rec[te])
        if len(set(yr))>1: aucs.append(roc_auc_score(yr,pr))
    return float(np.mean(aucs)), float(np.std(aucs))

def exp_b_mitigation(X,y,rec,db):
    base_m, base_s = lodo_with_optional_mitigation(X,y,rec,db, mitigate=False)
    mit_m,  mit_s  = lodo_with_optional_mitigation(X,y,rec,db, mitigate=True)
    pd.DataFrame([
        dict(setting="LODO baseline",            auc_mean=base_m, auc_std=base_s),
        dict(setting="LODO + per-source standardize", auc_mean=mit_m, auc_std=mit_s),
    ]).to_csv(f"{OUT}/mitigation.csv", index=False)
    print(f"\n[EXP B] LODO baseline           AUC = {base_m:.3f} ± {base_s:.3f}")
    print(f"[EXP B] LODO + per-source std   AUC = {mit_m:.3f} ± {mit_s:.3f}")
    print("  (improvement => leakage is partially mitigable; paper becomes constructive)")

# ============================================================================
if __name__ == "__main__":
    # PhysioNet must already be built into X,y,rec,db by your build_all().
    try:
        X, y, rec, db
    except NameError:
        raise SystemExit("Run build_all() first so X,y,rec,db exist, then re-run this cell.")
    Xtr_feat = np.concatenate([X.mean(axis=2), X.std(axis=2)], axis=1)

    # EXP A
    pascal = load_pascal(PASCAL_ROOT)
    exp_a_external(Xtr_feat, y, pascal)

    # EXP B
    exp_b_mitigation(X, y, rec, db)
    print("\nDONE. Upload: external_pascal.csv and mitigation.csv")

PASCAL_ROOT = /kaggle/input/datasets/kinguistics/heartbeat-sounds
PASCAL wavs found: 832
PASCAL segments: 1764 (normal=1084, abnormal=680)

[EXP A] Train PhysioNet -> Test PASCAL: AUC=0.458, bal_acc=0.485, n=1764
  (collapse here => the transfer failure is NOT specific to PhysioNet)


In [ ]:
# ============================================================
# خلية شاملة: بناء البيانات + تجربة CORAL
# ============================================================
import os, glob, wave
import numpy as np, pandas as pd
import scipy.linalg as sla
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

class CFG:
    raw_root="/kaggle/working/heart_sound"; prep_dir="/kaggle/working"
    sr=2000; win_sec=3.0; hop_sec=1.5; n_mels=40; n_fft=256; hop_fft=64
CFG.win=int(CFG.sr*CFG.win_sec); CFG.hop=int(CFG.sr*CFG.hop_sec); OUT=CFG.prep_dir

def read_wav(p):
    with wave.open(p,"rb") as wf:
        x=np.frombuffer(wf.readframes(wf.getnframes()),dtype=np.int16).astype(np.float32)
    return (x-x.mean())/(x.std()+1e-6) if x.std()>1e-6 else x

def build_all():
    wavs=glob.glob(os.path.join(CFG.raw_root,"**","*.wav"),recursive=True)
    print("[1] wav files:",len(wavs))
    rows={}
    for w in wavs:
        rec=os.path.splitext(os.path.basename(w))[0]; parts=w.lower().split(os.sep)
        if "unhealthy" in parts: lab=1
        elif "healthy" in parts: lab=0
        else: continue
        d=rec[0] if rec[:1].isalpha() else "?"
        rows.setdefault(rec,{"record":rec,"path":w,"db":d,"label":lab})
    man=pd.DataFrame(rows.values()).sort_values("record").reset_index(drop=True)
    print(f"[2] records: {len(man)} (healthy={(man.label==0).sum()}, unhealthy={(man.label==1).sum()})")
    segs,idx=[],[]
    for _,r in man.iterrows():
        x=read_wav(r["path"])
        if len(x)<CFG.win: x=np.pad(x,(0,CFG.win-len(x)))
        starts=list(range(0,len(x)-CFG.win+1,CFG.hop)) or [0]
        for s in starts:
            idx.append({"record":r["record"],"db":r["db"],"label":int(r["label"])})
            segs.append(x[s:s+CFG.win].astype(np.float16))
    seg_df=pd.DataFrame(idx); segs=np.stack(segs)
    print(f"[3] segments: {len(seg_df)}")
    cache=os.path.join(CFG.prep_dir,"mels.npy")
    if os.path.exists(cache):
        print("[4] loading cached mels"); M=np.load(cache)
    else:
        import torch, torchaudio
        mel=torchaudio.transforms.MelSpectrogram(sample_rate=CFG.sr,n_fft=CFG.n_fft,
            hop_length=CFG.hop_fft,n_mels=CFG.n_mels,power=2.0)
        todb=torchaudio.transforms.AmplitudeToDB(stype="power"); out=[]
        for i in range(0,len(segs),512):
            m=todb(mel(torch.from_numpy(segs[i:i+512].astype(np.float32))))
            m=(m-m.mean(dim=(1,2),keepdim=True))/(m.std(dim=(1,2),keepdim=True)+1e-6)
            out.append(m.numpy().astype(np.float16))
        M=np.concatenate(out); np.save(cache,M); print(f"[4] mel shape: {M.shape}")
    return M.astype(np.float32), seg_df["label"].values.astype(int), seg_df["record"].values, seg_df["db"].values

# ---- بناء ----
X,y,rec,db = build_all()
print(f"✓ جاهز: {len(y)} segments، {len(set(db))} قواعد\n")

# ---- CORAL ----
feats_all = np.concatenate([X.mean(axis=2), X.std(axis=2)], axis=1)

def to_record(p,ys,r):
    df=pd.DataFrame({"p":p,"y":ys,"r":r}); g=df.groupby("r").agg(p=("p","mean"),y=("y","first"))
    return g["p"].values,g["y"].values

def coral_transform(Xs,Xt,eps=1e-3):
    Xs=Xs.astype(np.float64); Xt=Xt.astype(np.float64)
    Cs=np.cov(Xs,rowvar=False)+eps*np.eye(Xs.shape[1])
    Ct=np.cov(Xt,rowvar=False)+eps*np.eye(Xt.shape[1])
    Ws=sla.fractional_matrix_power(Cs,-0.5).real
    Wt=sla.fractional_matrix_power(Ct, 0.5).real
    return (Xs-Xs.mean(0))@(Ws@Wt)+Xt.mean(0)

def lodo(adapt=False,clf_kind="rf"):
    rows=[]
    for held in sorted(set(db)):
        tr,te=db!=held,db==held
        if len(set(y[tr]))<2 or len(set(y[te]))<2: continue
        Xtr,Xte=feats_all[tr],feats_all[te]
        if adapt: Xtr=coral_transform(Xtr,Xte)
        sc=StandardScaler().fit(Xtr)
        clf=(RandomForestClassifier(n_estimators=200,n_jobs=-1) if clf_kind=="rf"
             else LogisticRegression(max_iter=1000,n_jobs=-1))
        clf.fit(sc.transform(Xtr),y[tr])
        p=clf.predict_proba(sc.transform(Xte))[:,1]
        pr,yr=to_record(p,y[te],rec[te])
        if len(set(yr))>1:
            rows.append(dict(held_out=held,auc=roc_auc_score(yr,pr),
                bal_acc=balanced_accuracy_score(yr,(pr>0.5).astype(int))))
    return pd.DataFrame(rows)

print("LODO بلا تكييف (RF)...")
base=lodo(False,"rf"); print(f"  mean AUC = {base.auc.mean():.3f} ± {base.auc.std():.3f}")
print("LODO + CORAL (RF)...")
coral=lodo(True,"rf"); print(f"  mean AUC = {coral.auc.mean():.3f} ± {coral.auc.std():.3f}")
print("LODO + CORAL (LogReg)...")
coral_lr=lodo(True,"lr"); print(f"  mean AUC = {coral_lr.auc.mean():.3f} ± {coral_lr.auc.std():.3f}")

summary=pd.DataFrame([
    dict(method="LODO no adaptation (RF)",auc_mean=base.auc.mean(),auc_std=base.auc.std()),
    dict(method="LODO + CORAL (RF)",auc_mean=coral.auc.mean(),auc_std=coral.auc.std()),
    dict(method="LODO + CORAL (LogReg)",auc_mean=coral_lr.auc.mean(),auc_std=coral_lr.auc.std()),
])
summary.to_csv(f"{OUT}/coral_lodo.csv",index=False)
print("\n",summary.round(3))
print("saved coral_lodo.csv")

In [ ]:
"""
Silhouette + UMAP to support the t-SNE (Reviewer: t-SNE is unstable).
Run AFTER build_all() so X, y, db exist. Fast.

Silhouette score quantifies cluster separation (range -1..1):
  - by SOURCE database: high => features cluster strongly by source (leakage)
  - by CLASS (pathology): low/near-0 => features do NOT separate by pathology
This gives a STABLE, quantitative complement to the (parameter-dependent) t-SNE.
"""
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
OUT = "/kaggle/working"

feats = np.concatenate([X.mean(axis=2), X.std(axis=2)], axis=1)
# subsample for speed/stability
rng = np.random.default_rng(0)
i = rng.choice(len(feats), min(5000, len(feats)), replace=False)
F = StandardScaler().fit_transform(feats[i]); D = np.array(db)[i]; Y = y[i]

sil_source = silhouette_score(F, D, metric="euclidean", sample_size=5000, random_state=0)
sil_class  = silhouette_score(F, Y, metric="euclidean", sample_size=5000, random_state=0)
print(f"Silhouette by SOURCE database = {sil_source:.3f}  (higher => stronger source clustering)")
print(f"Silhouette by CLASS (pathology) = {sil_class:.3f}  (near 0 => no pathology clustering)")

rows = [dict(grouping="source_database", silhouette=sil_source),
        dict(grouping="class_pathology", silhouette=sil_class)]

# Optional UMAP replication of the t-SNE (if umap-learn is available)
try:
    import umap
    reducer = umap.UMAP(n_neighbors=30, min_dist=0.1, random_state=0)
    emb = reducer.fit_transform(F)
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    for d in sorted(set(D)):
        m = D == d; ax[0].scatter(emb[m,0], emb[m,1], s=4, label=f"DB {d}", alpha=0.5)
    ax[0].legend(markerscale=3, fontsize=8); ax[0].set_title("UMAP by SOURCE DATABASE")
    for c,lab in [(0,"normal"),(1,"abnormal")]:
        m = Y == c; ax[1].scatter(emb[m,0], emb[m,1], s=4, label=lab, alpha=0.5)
    ax[1].legend(markerscale=3, fontsize=8); ax[1].set_title("UMAP by CLASS")
    plt.tight_layout(); fig.savefig(f"{OUT}/fig_umap_embeddings.png", dpi=300)
    print("saved fig_umap_embeddings.png (UMAP replication of t-SNE)")
except ImportError:
    print("umap-learn not installed; silhouette scores alone are sufficient.")
    print("  (to install: !pip install umap-learn)")

pd.DataFrame(rows).to_csv(f"{OUT}/silhouette_scores.csv", index=False)
print("\nsaved silhouette_scores.csv")
print("\nINTERPRETATION: silhouette(source) >> silhouette(class) confirms, quantitatively")
print("and stably, what the t-SNE shows: features organize by source, not pathology.")

In [ ]:
import glob, os, subprocess

# تحقّقي: هل ملفات wav موجودة؟
wavs = glob.glob("/kaggle/working/heart_sound/**/*.wav", recursive=True)
print("ملفات wav موجودة:", len(wavs))

if len(wavs) == 0:
    print("جاري التحميل...")
    r = subprocess.run(
        "kaggle datasets download -d swapnilpanda/heart-sound-database -p /kaggle/working --unzip",
        shell=True, capture_output=True, text=True)
    print("stderr:", r.stderr[-300:])
    wavs = glob.glob("/kaggle/working/heart_sound/**/*.wav", recursive=True)
    print("بعد التحميل:", len(wavs), "ملف")

# تحقّقي من المسار والبنية
if wavs:
    print("أمثلة:", wavs[:3])
    # تأكّدي من وجود healthy/unhealthy في المسار
    sample = wavs[0].lower()
    print("يحتوي healthy/unhealthy؟", 'healthy' in sample or 'unhealthy' in sample)

In [ ]:
# ============================================================
# خلية شاملة: بناء البيانات + within-database (أولوية 2+3)
# ============================================================
import os, glob, wave
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

class CFG:
    raw_root="/kaggle/working/heart_sound"; prep_dir="/kaggle/working"
    sr=2000; win_sec=3.0; hop_sec=1.5; n_mels=40; n_fft=256; hop_fft=64
CFG.win=int(CFG.sr*CFG.win_sec); CFG.hop=int(CFG.sr*CFG.hop_sec); OUT=CFG.prep_dir

def read_wav(p):
    with wave.open(p,"rb") as wf:
        x=np.frombuffer(wf.readframes(wf.getnframes()),dtype=np.int16).astype(np.float32)
    return (x-x.mean())/(x.std()+1e-6) if x.std()>1e-6 else x

def build_all():
    wavs=glob.glob(os.path.join(CFG.raw_root,"**","*.wav"),recursive=True)
    print("[1] wav files:",len(wavs))
    rows={}
    for w in wavs:
        r=os.path.splitext(os.path.basename(w))[0]; parts=w.lower().split(os.sep)
        if "unhealthy" in parts: lab=1
        elif "healthy" in parts: lab=0
        else: continue
        d=r[0] if r[:1].isalpha() else "?"
        rows.setdefault(r,{"record":r,"path":w,"db":d,"label":lab})
    man=pd.DataFrame(rows.values()).sort_values("record").reset_index(drop=True)
    print(f"[2] records: {len(man)} (healthy={(man.label==0).sum()}, unhealthy={(man.label==1).sum()})")
    segs,idx=[],[]
    for _,r in man.iterrows():
        x=read_wav(r["path"])
        if len(x)<CFG.win: x=np.pad(x,(0,CFG.win-len(x)))
        for s in (list(range(0,len(x)-CFG.win+1,CFG.hop)) or [0]):
            idx.append({"record":r["record"],"db":r["db"],"label":int(r["label"])})
            segs.append(x[s:s+CFG.win].astype(np.float16))
    seg_df=pd.DataFrame(idx); segs=np.stack(segs)
    print(f"[3] segments: {len(seg_df)}")
    cache=os.path.join(CFG.prep_dir,"mels.npy")
    if os.path.exists(cache):
        print("[4] loading cached mels"); M=np.load(cache)
    else:
        import torch, torchaudio
        mel=torchaudio.transforms.MelSpectrogram(sample_rate=CFG.sr,n_fft=CFG.n_fft,
            hop_length=CFG.hop_fft,n_mels=CFG.n_mels,power=2.0)
        todb=torchaudio.transforms.AmplitudeToDB(stype="power"); out=[]
        for i in range(0,len(segs),512):
            m=todb(mel(torch.from_numpy(segs[i:i+512].astype(np.float32))))
            m=(m-m.mean(dim=(1,2),keepdim=True))/(m.std(dim=(1,2),keepdim=True)+1e-6)
            out.append(m.numpy().astype(np.float16))
        M=np.concatenate(out); np.save(cache,M); print(f"[4] mel shape: {M.shape}")
    return M.astype(np.float32), seg_df["label"].values.astype(int), seg_df["record"].values, seg_df["db"].values

# ---- بناء ----
X,y,rec,db = build_all()
print(f"✓ جاهز: {len(y)} segments، {len(set(db))} قواعد {sorted(set(db))}\n")

# ---- within-database ----
feats = np.concatenate([X.mean(axis=2), X.std(axis=2)], axis=1)

def to_record(p,ys,r):
    df=pd.DataFrame({"p":p,"y":ys,"r":r}); g=df.groupby("r").agg(p=("p","mean"),y=("y","first"))
    return g["p"].values,g["y"].values

def record_split(mask, seed=42, frac=0.7):
    recs=np.array(sorted(set(rec[mask]))); rng=np.random.default_rng(seed); rng.shuffle(recs)
    tr_recs=set(recs[:int(frac*len(recs))]); idx=np.where(mask)[0]
    tr=np.array([rec[i] in tr_recs for i in idx]); return idx[tr], idx[~tr]

rows=[]
print("=== WITHIN-DATABASE (train & test inside each DB, record-level) ===")
for d in sorted(set(db)):
    mask=db==d; yd=y[mask]
    n_pos,n_neg=int((yd==1).sum()),int((yd==0).sum())
    if n_pos<10 or n_neg<10:
        rows.append(dict(database=d,within_auc=np.nan,n_pos=n_pos,n_neg=n_neg,note="too few of one class"))
        print(f"  DB {d}: skipped (pos={n_pos}, neg={n_neg})"); continue
    aucs=[]
    for seed in [42,123,2024]:
        tr,te=record_split(mask,seed=seed)
        if len(set(y[tr]))<2 or len(set(y[te]))<2: continue
        sc=StandardScaler().fit(feats[tr])
        clf=RandomForestClassifier(n_estimators=200,n_jobs=-1,random_state=seed).fit(sc.transform(feats[tr]),y[tr])
        p=clf.predict_proba(sc.transform(feats[te]))[:,1]
        pr,yr=to_record(p,y[te],rec[te])
        if len(set(yr))>1: aucs.append(roc_auc_score(yr,pr))
    m=float(np.mean(aucs)) if aucs else np.nan
    sd=float(np.std(aucs)) if aucs else np.nan
    rows.append(dict(database=d,within_auc=m,within_std=sd,n_pos=n_pos,n_neg=n_neg))
    print(f"  DB {d}: within-AUC = {m:.3f} ± {sd:.3f}  (pos={n_pos}, neg={n_neg})")

within=pd.DataFrame(rows); within.to_csv(f"{OUT}/within_database.csv",index=False)
valid=within.within_auc.dropna()
print(f"\n[within-DB] mean={valid.mean():.3f}, range={valid.min():.3f}-{valid.max():.3f}")
print("INTERPRETATION: high within-DB + near-chance LODO => signal exists locally, doesn't transfer.")
print("\nsaved within_database.csv")

In [ ]:
# ============ بناء البيانات + DANN تحت LODO ============
import os, glob, wave, numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score
torch.manual_seed(0); np.random.seed(0)

class CFG:
    raw_root="/kaggle/working/heart_sound"; prep_dir="/kaggle/working"
    sr=2000; win_sec=3.0; hop_sec=1.5; n_mels=40; n_fft=256; hop_fft=64
CFG.win=int(CFG.sr*CFG.win_sec); CFG.hop=int(CFG.sr*CFG.hop_sec); OUT=CFG.prep_dir

def read_wav(p):
    with wave.open(p,"rb") as wf:
        x=np.frombuffer(wf.readframes(wf.getnframes()),dtype=np.int16).astype(np.float32)
    return (x-x.mean())/(x.std()+1e-6) if x.std()>1e-6 else x

def build_all():
    wavs=glob.glob(os.path.join(CFG.raw_root,"**","*.wav"),recursive=True)
    print("[1] wav files:",len(wavs))
    rows={}
    for w in wavs:
        r=os.path.splitext(os.path.basename(w))[0]; parts=w.lower().split(os.sep)
        if "unhealthy" in parts: lab=1
        elif "healthy" in parts: lab=0
        else: continue
        d=r[0] if r[:1].isalpha() else "?"
        rows.setdefault(r,{"record":r,"path":w,"db":d,"label":lab})
    man=pd.DataFrame(list(rows.values())).sort_values("record").reset_index(drop=True)
    print(f"[2] records: {len(man)}")
    segs,idx=[],[]
    for _,r in man.iterrows():
        x=read_wav(r["path"])
        if len(x)<CFG.win: x=np.pad(x,(0,CFG.win-len(x)))
        for s in (list(range(0,len(x)-CFG.win+1,CFG.hop)) or [0]):
            idx.append({"record":r["record"],"db":r["db"],"label":int(r["label"])})
            segs.append(x[s:s+CFG.win].astype(np.float16))
    seg_df=pd.DataFrame(idx); segs=np.stack(segs)
    print(f"[3] segments: {len(seg_df)}")
    cache=os.path.join(CFG.prep_dir,"mels.npy")
    if os.path.exists(cache):
        M=np.load(cache)
        if len(M)!=len(segs): os.remove(cache); M=None
        else: print("[4] loaded cached mels")
    else: M=None
    if M is None:
        import torchaudio
        mel=torchaudio.transforms.MelSpectrogram(sample_rate=CFG.sr,n_fft=CFG.n_fft,
            hop_length=CFG.hop_fft,n_mels=CFG.n_mels,power=2.0)
        todb=torchaudio.transforms.AmplitudeToDB(stype="power"); out=[]
        for i in range(0,len(segs),512):
            m=todb(mel(torch.from_numpy(segs[i:i+512].astype(np.float32))))
            m=(m-m.mean(dim=(1,2),keepdim=True))/(m.std(dim=(1,2),keepdim=True)+1e-6)
            out.append(m.numpy().astype(np.float16))
        M=np.concatenate(out); np.save(cache,M); print(f"[4] mels {M.shape}")
    return M.astype(np.float32), seg_df["label"].values.astype(int), seg_df["record"].values, seg_df["db"].values

X,y,rec,db = build_all()
print(f"✓ جاهز: {len(y)} segments\n")

# ---- DANN ----
Xt=torch.tensor(X[:,None,:,:],dtype=torch.float32); y_t=torch.tensor(y,dtype=torch.float32)
dbs=sorted(set(db)); db2i={d:i for i,d in enumerate(dbs)}
dom=torch.tensor([db2i[d] for d in db],dtype=torch.long)

class GR(torch.autograd.Function):
    @staticmethod
    def forward(c,x,l): c.l=l; return x.view_as(x)
    @staticmethod
    def backward(c,g): return g.neg()*c.l, None
def grl(x,l): return GR.apply(x,l)

class DANN(nn.Module):
    def __init__(s,nd):
        super().__init__()
        s.f=nn.Sequential(nn.Conv2d(1,16,3,padding=1),nn.BatchNorm2d(16),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),nn.AdaptiveAvgPool2d(1),nn.Flatten())
        s.c=nn.Linear(64,1); s.d=nn.Sequential(nn.Linear(64,32),nn.ReLU(),nn.Linear(32,nd))
    def forward(s,x,l=0.0):
        z=s.f(x); return s.c(z).squeeze(1), s.d(grl(z,l))

def to_record(p,ys,r):
    d=pd.DataFrame({"p":p,"y":ys,"r":r}); g=d.groupby("r").agg(p=("p","mean"),y=("y","first"))
    return g["p"].values,g["y"].values

def run(use_dann,epochs=8):
    rows=[]
    for held in dbs:
        tr=db!=held; te=db==held
        if len(set(y[tr]))<2 or len(set(y[te]))<2: continue
        tdoms=sorted(set(dom[tr].tolist())); dr={d:i for i,d in enumerate(tdoms)}
        dtr=torch.tensor([dr[int(v)] for v in dom[tr]],dtype=torch.long)
        dl=DataLoader(TensorDataset(Xt[tr],y_t[tr],dtr),batch_size=256,shuffle=True)
        net=DANN(len(tdoms)); opt=torch.optim.Adam(net.parameters(),lr=1e-3)
        pw=torch.tensor([(y[tr]==0).sum()/max(1,(y[tr]==1).sum())],dtype=torch.float32)
        bce=nn.BCEWithLogitsLoss(pos_weight=pw); ce=nn.CrossEntropyLoss()
        for ep in range(epochs):
            lam=(2/(1+np.exp(-10*ep/epochs))-1) if use_dann else 0.0
            for xb,yb,dbb in dl:
                opt.zero_grad(); cl,do=net(xb,lam); loss=bce(cl,yb)
                if use_dann: loss=loss+ce(do,dbb)
                loss.backward(); opt.step()
        net.eval()
        with torch.no_grad():
            lg,_=net(Xt[te],0.0); p=torch.sigmoid(lg).numpy()
        pr,yr=to_record(p,y[te],rec[te])
        if len(set(yr))>1:
            a=roc_auc_score(yr,pr); rows.append(dict(held_out=held,auc=a))
            print(f"  {'DANN' if use_dann else 'plain'} held {held}: AUC={a:.3f}")
    return pd.DataFrame(rows)

print("=== plain CNN (LODO) ==="); base=run(False)
print(f"  plain mean = {base.auc.mean():.3f} ± {base.auc.std():.3f}\n")
print("=== DANN (LODO) ==="); dann=run(True)
print(f"  DANN mean = {dann.auc.mean():.3f} ± {dann.auc.std():.3f}\n")
pd.DataFrame([
    dict(method="plain CNN (LODO)",auc_mean=base.auc.mean(),auc_std=base.auc.std()),
    dict(method="DANN (LODO)",auc_mean=dann.auc.mean(),auc_std=dann.auc.std()),
]).to_csv(f"{OUT}/dann_lodo.csv",index=False)
print("saved dann_lodo.csv")

In [ ]:
"""
Graphical Abstract — Heart Sound Performance-Inflation paper (BSPC)
LARGE, BOLD, high-contrast text for readability at 5x13 cm.
Run on Kaggle (CPU). Your own matplotlib code (not AI image generation).
Output: graphical_abstract.png at 300 dpi.
"""
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import numpy as np

plt.rcParams['font.family'] = 'DejaVu Sans'

INK   = '#0F0F23'
LEAK  = '#1A56DB'   # strong blue
BAD   = '#D32030'   # strong red
GOOD  = '#0E7A6E'   # strong teal
SOFT  = '#3A3A4A'   # darker grey for contrast
PANEL = '#EEF1F8'

fig, ax = plt.subplots(figsize=(13, 6.4), dpi=300)
ax.set_xlim(0, 13); ax.set_ylim(0, 6.4); ax.axis('off')
fig.patch.set_facecolor('white')

def heartbeat(ax, x0, y0, w, h, color, lw=3.0):
    n=400; x=np.linspace(0,1,n); y=np.zeros(n)
    y += 1.0*np.exp(-((x-0.45)/0.012)**2)
    y += -0.35*np.exp(-((x-0.49)/0.015)**2)
    y += 0.30*np.exp(-((x-0.72)/0.04)**2)
    ax.plot(x0+x*w, y0+y*h, color=color, lw=lw, solid_capstyle='round', zorder=5)

# ===== TITLE (larger, bolder) =====
ax.text(6.5, 6.05, 'Same model and data \u2014 only the evaluation split changes',
        ha='center', va='center', fontsize=20, fontweight='bold', color=INK)
ax.text(6.5, 5.55, 'Heart sound (PCG) classification on PhysioNet/CinC 2016',
        ha='center', va='center', fontsize=13, color=SOFT, fontweight='bold')

# ===== THREE CARDS =====
cards=[(0.6,'SEGMENT-LEVEL','leakage-prone','0.978',LEAK),
       (4.75,'RECORD-LEVEL','assumed safe','0.964',LEAK),
       (8.9,'CROSS-DATABASE','source-independent','0.503',BAD)]
cw,ch,cy=3.55,2.35,2.55
for x,title,sub,auc,color in cards:
    ax.add_patch(FancyBboxPatch((x+0.05,cy-0.07),cw,ch,boxstyle="round,pad=0.10",
                 linewidth=0,facecolor='#00000015',zorder=1))
    ax.add_patch(FancyBboxPatch((x,cy),cw,ch,boxstyle="round,pad=0.10",
                 linewidth=3.2,edgecolor=color,facecolor='white',zorder=2))
    ax.add_patch(FancyBboxPatch((x,cy+ch-0.62),cw,0.62,boxstyle="round,pad=0.10",
                 linewidth=0,facecolor=color,zorder=3))
    ax.text(x+cw/2,cy+ch-0.31,title,ha='center',va='center',
            fontsize=15,fontweight='bold',color='white',zorder=4)
    ax.text(x+cw/2,cy+ch-0.92,sub,ha='center',va='center',
            fontsize=12,color=SOFT,fontweight='bold',zorder=4)
    heartbeat(ax,x+0.5,cy+0.92,cw-1.0,0.5,color)
    ax.text(x+cw/2,cy+0.46,'record-level AUC',ha='center',va='center',
            fontsize=11.5,color=SOFT,fontweight='bold',zorder=4)
    ax.text(x+cw/2,cy+0.02,auc,ha='center',va='center',
            fontsize=34,fontweight='bold',color=color,zorder=4)

for xs in [4.32,8.47]:
    ax.add_patch(FancyArrowPatch((xs,cy+ch/2),(xs+0.38,cy+ch/2),
                 arrowstyle='-|>',mutation_scale=34,linewidth=4,color=SOFT,zorder=2))

ax.text(8.9+cw/2,cy-0.34,'\u25BC  near-chance (0.5)',ha='center',va='center',
        fontsize=14,fontweight='bold',color=BAD)

# ===== MECHANISM PANEL =====
py,ph=0.35,1.65
ax.add_patch(FancyBboxPatch((0.6,py),11.8,ph,boxstyle="round,pad=0.12",
             linewidth=0,facecolor=PANEL,zorder=0))
ax.text(1.1,py+ph-0.36,'WHY THE COLLAPSE?',ha='left',va='center',
        fontsize=15,fontweight='bold',color=INK)
ax.text(1.1,py+0.52,'The model learns the\nrecording source,\nnot the pathology',
        ha='left',va='center',fontsize=13,fontweight='bold',color=BAD)

metrics=[(5.1,'Source-identity\nprobe','0.997',BAD),
         (8.0,'Within-database\nmean','0.871',GOOD),
         (10.9,'External\n(PASCAL)','0.447',BAD)]
for x,label,val,color in metrics:
    ax.text(x,py+ph-0.40,label,ha='center',va='center',fontsize=13,fontweight='bold',color=INK)
    ax.text(x,py+0.48,val,ha='center',va='center',fontsize=26,fontweight='bold',color=color)
    if x<10:
        ax.plot([x+1.45,x+1.45],[py+0.25,py+ph-0.25],color='#C5CDDE',lw=1.5)

plt.subplots_adjust(left=0.01,right=0.99,top=0.99,bottom=0.01)
plt.savefig('graphical_abstract.png',dpi=300,bbox_inches='tight',facecolor='white',pad_inches=0.18)
print("Saved graphical_abstract.png (300 dpi)")

In [ ]:
import shutil, os

# احذف الـ zip القديم
if os.path.exists('/kaggle/working/my_work.zip'):
    os.remove('/kaggle/working/my_work.zip')

# انسخ الملفّات المهمّة فقط (بلا mels.npy الضخم وبلا مجلّدات البيانات)
os.makedirs('/kaggle/upload', exist_ok=True)
import glob
for f in glob.glob('/kaggle/working/*.csv') + glob.glob('/kaggle/working/*.png') + glob.glob('/kaggle/working/*.py') + glob.glob('/kaggle/working/*.ipynb'):
    shutil.copy(f, '/kaggle/upload/')

# اضغط فقط الملفّات المهمّة
shutil.make_archive('/kaggle/my_work', 'zip', '/kaggle/upload')
shutil.move('/kaggle/my_work.zip', '/kaggle/working/my_work.zip')
print("✓ تمّ! الحجم:", round(os.path.getsize('/kaggle/working/my_work.zip')/1024/1024, 2), "MB")
print("حمّلي my_work.zip من Output")

In [ ]:
# =============================================================================
#  HEART SOUND LEAKAGE AUDIT - COMPLETE REPRODUCIBLE PIPELINE
#  ---------------------------------------------------------------------------
#  A quantitative audit of evaluation-protocol-induced performance inflation
#  in heart sound (PCG) classification on the PhysioNet/CinC 2016 dataset.
#
#  Three evaluation protocols are compared using an identical lightweight CNN:
#    (1) segment-level leakage   - common, flawed (segments shuffled randomly)
#    (2) record-level split      - assumed-clean (split by recording)
#    (3) cross-database split    - true generalization (whole databases held out)
#
#  Run order: this single file executes the full study end to end.
#  Environment: Kaggle notebook (CPU is sufficient; GPU optional).
# =============================================================================

import os, glob, time, json, wave
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.autograd import Function
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (roc_auc_score, f1_score,
                             balanced_accuracy_score, accuracy_score)
from scipy import stats

# -----------------------------------------------------------------------------
# CONFIGURATION
# -----------------------------------------------------------------------------
class CFG:
    raw_root  = "/kaggle/working/heart_sound"      # raw wav files
    prep_dir  = "/kaggle/working/prepared"         # cached arrays / index
    res_dir   = "/kaggle/working/results"          # metrics
    fig_dir   = "/kaggle/working/figures"          # figures
    kaggle_ds = "swapnilpanda/heart-sound-database"

    sr        = 2000            # sampling rate (Hz), uniform across dataset
    win_sec   = 3.0            # segment window length (s)
    hop_sec   = 1.5            # hop between segments (50% overlap)
    n_mels    = 40            # mel bands
    n_fft     = 256
    hop_fft   = 64

    epochs    = 8
    patience  = 3
    batch     = 256
    lr        = 1e-3
    seeds     = [42, 123, 2024, 7, 99]            # 5 seeds for significance

    # cross-database split: which databases go to train / val / test
    db_split  = {"e": "train", "a": "train", "b": "val",
                 "c": "test", "d": "test", "f": "test"}

    protocols = ["leaky_segment", "clean_record", "clean_database"]

CFG.win = int(CFG.sr * CFG.win_sec)
CFG.hop = int(CFG.sr * CFG.hop_sec)
for d in [CFG.prep_dir, CFG.res_dir, CFG.fig_dir]:
    os.makedirs(d, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(os.cpu_count())
print(f"Device: {DEVICE}")


# =============================================================================
# STAGE 1 - DATA ACQUISITION
# =============================================================================
def acquire_data():
    """Download the raw PCG dataset from Kaggle if not already present."""
    wavs = glob.glob(os.path.join(CFG.raw_root, "**", "*.wav"), recursive=True)
    if len(wavs) == 0:
        print("[1] Downloading dataset from Kaggle ...")
        os.system(f"kaggle datasets download -d {CFG.kaggle_ds} "
                  f"-p /kaggle/working --unzip")
        wavs = glob.glob(os.path.join(CFG.raw_root, "**", "*.wav"), recursive=True)
    print(f"[1] Raw wav files available: {len(wavs)}")
    return wavs


# =============================================================================
# STAGE 2 - BUILD RECORD MANIFEST
#   Ignore any pre-existing train/val split (it is leakage-contaminated).
#   Label is taken from folder name: healthy=0, unhealthy=1.
#   Database letter (a-f) is the first character of the record id.
# =============================================================================
def build_manifest(wavs):
    print("[2] Building unique-record manifest ...")
    rows = {}
    for w in wavs:
        rec = os.path.splitext(os.path.basename(w))[0]
        parts = w.lower().split(os.sep)
        if   "unhealthy" in parts: label = 1
        elif "healthy"   in parts: label = 0
        else: continue
        db = rec[0] if rec[:1].isalpha() else "?"
        if rec not in rows:
            rows[rec] = {"record": rec, "path": w, "db": db, "label": label}
    man = pd.DataFrame(list(rows.values())).sort_values("record").reset_index(drop=True)
    print(f"    unique records: {len(man)} "
          f"(healthy={int((man.label==0).sum())}, "
          f"unhealthy={int((man.label==1).sum())})")
    return man


# =============================================================================
# STAGE 3 - SEGMENTATION
#   Each recording -> fixed-length overlapping windows.
#   Per-recording z-score normalization. Provenance kept for every segment.
# =============================================================================
def read_wav(path):
    with wave.open(path, "rb") as wf:
        x = np.frombuffer(wf.readframes(wf.getnframes()), dtype=np.int16).astype(np.float32)
    return (x - x.mean()) / (x.std() + 1e-6) if x.std() > 1e-6 else x

def segment_signals(man):
    print("[3] Segmenting recordings ...")
    segs, idx = [], []
    for _, r in man.iterrows():
        x = read_wav(r["path"])
        if len(x) < CFG.win:
            x = np.pad(x, (0, CFG.win - len(x)))
        starts = list(range(0, len(x) - CFG.win + 1, CFG.hop)) or [0]
        for s in starts:
            idx.append({"seg_id": len(segs), "record": r["record"],
                        "db": r["db"], "label": int(r["label"])})
            segs.append(x[s:s + CFG.win].astype(np.float16))
    segs = np.stack(segs)
    seg_df = pd.DataFrame(idx)
    print(f"    total segments: {len(seg_df)} "
          f"(avg {len(seg_df)/len(man):.1f} per recording)")
    return segs, seg_df


# =============================================================================
# STAGE 4 - LOG-MEL SPECTROGRAMS (precomputed once)
# =============================================================================
def compute_mels(segs):
    cache = os.path.join(CFG.prep_dir, "mels.npy")
    if os.path.exists(cache):
        print("[4] Loading cached mel-spectrograms ...")
        return np.load(cache)
    print("[4] Computing log-mel spectrograms ...")
    import torchaudio
    mel = torchaudio.transforms.MelSpectrogram(
        sample_rate=CFG.sr, n_fft=CFG.n_fft, hop_length=CFG.hop_fft,
        n_mels=CFG.n_mels, power=2.0)
    todb = torchaudio.transforms.AmplitudeToDB(stype="power")
    out = []
    for i in range(0, len(segs), 512):
        m = todb(mel(torch.from_numpy(segs[i:i+512].astype(np.float32))))
        m = (m - m.mean(dim=(1,2), keepdim=True)) / (m.std(dim=(1,2), keepdim=True) + 1e-6)
        out.append(m.numpy().astype(np.float16))
    mels = np.concatenate(out)
    np.save(cache, mels)
    print(f"    mel shape: {mels.shape}")
    return mels


# =============================================================================
# STAGE 5 - DEFINE THE THREE EVALUATION PROTOCOLS
#   All use a 70/15/15 train/val/test ratio; only the split UNIT differs.
# =============================================================================
def assign_by_unit(units, frac=(0.70, 0.15, 0.15), seed=42):
    u = list(units); np.random.default_rng(seed).shuffle(u)
    n = len(u); a = int(frac[0]*n); b = int(frac[1]*n)
    out = {}
    for x in u[:a]:        out[x] = "train"
    for x in u[a:a+b]:     out[x] = "val"
    for x in u[a+b:]:      out[x] = "test"
    return out

def build_protocols(seg_df, man):
    print("[5] Building the three evaluation protocols ...")
    seg_df = seg_df.copy()
    # (1) segment-level leakage: shuffle segments directly
    seg_df["split_leaky_segment"] = seg_df["seg_id"].map(
        assign_by_unit(seg_df["seg_id"].tolist()))
    # (2) record-level: split by recording id
    seg_df["split_clean_record"] = seg_df["record"].map(
        assign_by_unit(man["record"].tolist()))
    # (3) cross-database: whole databases held out
    seg_df["split_clean_database"] = seg_df["db"].map(
        lambda d: CFG.db_split.get(d, "train"))

    # integrity check: count records that leak across splits
    print("    leakage integrity check (records appearing in >1 split):")
    for p in CFG.protocols:
        col = f"split_{p}"
        leaked = (seg_df.groupby("record")[col].nunique() > 1).sum()
        flag = "expected (flawed)" if p == "leaky_segment" else "clean"
        print(f"      {p:16s}: {leaked:4d}  [{flag}]")
    seg_df.to_csv(os.path.join(CFG.prep_dir, "seg_index_with_splits.csv"), index=False)
    return seg_df


# =============================================================================
# STAGE 6 - MODEL (lightweight CNN, ~23.6k params)
#   Optional domain-adversarial head for the mitigation experiment.
# =============================================================================
class GradReverse(Function):
    @staticmethod
    def forward(ctx, x, lam): ctx.lam = lam; return x.view_as(x)
    @staticmethod
    def backward(ctx, g): return g.neg() * ctx.lam, None

class LightCNN(nn.Module):
    def __init__(self, n_db=0):
        super().__init__()
        self.feat = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1))
        self.cls = nn.Linear(64, 1)
        self.dom = (nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, n_db))
                    if n_db > 0 else None)
    def forward(self, x, lam=0.0):
        f = self.feat(x).flatten(1)
        logit = self.cls(f).squeeze(1)
        if self.dom is not None:
            return logit, self.dom(GradReverse.apply(f, lam))
        return logit


# =============================================================================
# STAGE 7 - DATASET / EVALUATION HELPERS
# =============================================================================
class MelDS(Dataset):
    def __init__(self, mels, idx):
        self.mels = mels
        self.sid = idx["seg_id"].values
        self.y   = idx["label"].values.astype(np.float32)
    def __len__(self): return len(self.sid)
    def __getitem__(self, i):
        s = self.sid[i]
        return torch.from_numpy(self.mels[s].astype(np.float32)).unsqueeze(0), self.y[i], int(s)

def metrics(y, p):
    yh = (p >= 0.5).astype(int)
    return dict(auc=roc_auc_score(y, p) if len(np.unique(y)) > 1 else np.nan,
                f1=f1_score(y, yh, zero_division=0),
                bal_acc=balanced_accuracy_score(y, yh),
                acc=accuracy_score(y, yh))

def evaluate(model, loader, seg_df):
    """Return both record-level (clinical) and segment-level metrics."""
    model.eval(); P, Y, S = [], [], []
    with torch.no_grad():
        for xb, yb, sid in loader:
            out = model(xb.to(DEVICE))
            if isinstance(out, tuple): out = out[0]
            P.append(torch.sigmoid(out).cpu().numpy()); Y.append(yb.numpy()); S.append(sid.numpy())
    P, Y, S = np.concatenate(P), np.concatenate(Y), np.concatenate(S)
    seg_m = metrics(Y, P)
    t = pd.DataFrame({"seg_id": S, "prob": P}).merge(
        seg_df[["seg_id", "record", "label"]], on="seg_id")
    rec = t.groupby("record").agg(prob=("prob", "mean"), label=("label", "first")).reset_index()
    rec_m = metrics(rec["label"].values, rec["prob"].values)
    return rec_m, seg_m


# =============================================================================
# STAGE 8 - TRAIN ONE (protocol, seed)
# =============================================================================
def train_one(mels, seg_df, protocol, seed):
    np.random.seed(seed); torch.manual_seed(seed)
    col = f"split_{protocol}"
    tr = seg_df[seg_df[col] == "train"]
    va = seg_df[seg_df[col] == "val"]
    te = seg_df[seg_df[col] == "test"]
    trL = DataLoader(MelDS(mels, tr), CFG.batch, shuffle=True,  num_workers=2)
    vaL = DataLoader(MelDS(mels, va), CFG.batch, shuffle=False, num_workers=2)
    teL = DataLoader(MelDS(mels, te), CFG.batch, shuffle=False, num_workers=2)

    model = LightCNN().to(DEVICE)
    pos_w = torch.tensor([(tr.label == 0).sum() / max(1, (tr.label == 1).sum())], device=DEVICE)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    opt  = torch.optim.Adam(model.parameters(), lr=CFG.lr)

    best, best_state, bad = -1, None, 0
    for ep in range(CFG.epochs):
        model.train()
        for xb, yb, _ in trL:
            opt.zero_grad(); crit(model(xb.to(DEVICE)), yb.to(DEVICE)).backward(); opt.step()
        rec_m, _ = evaluate(model, vaL, seg_df)
        if not np.isnan(rec_m["auc"]) and rec_m["auc"] > best:
            best, best_state, bad = rec_m["auc"], {k: v.cpu().clone() for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= CFG.patience: break
    if best_state: model.load_state_dict(best_state)
    n_params = sum(p.numel() for p in model.parameters())
    rec_m, seg_m = evaluate(model, teL, seg_df)
    return rec_m, seg_m, n_params


# =============================================================================
# STAGE 9 - RUN ALL PROTOCOLS x SEEDS  +  SIGNIFICANCE TESTS
# =============================================================================
def run_study(mels, seg_df):
    print("[9] Training all protocols x seeds ...")
    rows = []; t0 = time.time()
    for proto in CFG.protocols:
        for sd in CFG.seeds:
            rec_m, seg_m, npar = train_one(mels, seg_df, proto, sd)
            rows.append({"protocol": proto, "seed": sd, "eval": "record", "params": npar, **rec_m})
            rows.append({"protocol": proto, "seed": sd, "eval": "segment", "params": npar, **seg_m})
            pd.DataFrame(rows).to_csv(os.path.join(CFG.res_dir, "final_results.csv"), index=False)
            print(f"    {proto:16s} seed={sd:<4d} "
                  f"REC auc={rec_m['auc']:.3f} | SEG auc={seg_m['auc']:.3f}")
    res = pd.DataFrame(rows)
    print(f"    elapsed: {(time.time()-t0)/60:.1f} min")

    print("\n[9] Summary (record-level, mean +/- sd over seeds):")
    rec = res[res["eval"] == "record"]
    for p in CFG.protocols:
        a = rec[rec.protocol == p]["auc"]
        print(f"    {p:16s}: AUC = {a.mean():.3f} +/- {a.std():.3f}")

    print("\n[9] Paired significance tests (record-level AUC):")
    def auc(p): return rec[rec.protocol == p].sort_values("seed")["auc"].values
    for a, b in [("leaky_segment","clean_record"),
                 ("clean_record","clean_database"),
                 ("leaky_segment","clean_database")]:
        x, y = auc(a), auc(b)
        tp = stats.ttest_rel(x, y).pvalue
        try: wp = stats.wilcoxon(x, y).pvalue
        except Exception: wp = np.nan
        print(f"    {a} vs {b}: diff={x.mean()-y.mean():+.3f} "
              f"t_p={tp:.4f} wilcoxon_p={wp:.4f}")
    return res


# =============================================================================
# MAIN
# =============================================================================
def main():
    wavs   = acquire_data()
    man    = build_manifest(wavs)
    man.to_csv(os.path.join(CFG.prep_dir, "manifest_records.csv"), index=False)
    segs, seg_df = segment_signals(man)
    mels   = compute_mels(segs)
    seg_df = build_protocols(seg_df, man)
    res    = run_study(mels, seg_df)
    print("\nPipeline complete. Results in /kaggle/working/results/final_results.csv")
    print("Next: run the figure script (08_make_figures_en.py) to regenerate plots.")

if __name__ == "__main__":
    main()

In [ ]:
"""
خلية كشف تلقائي — شغّلها على Kaggle لإيجاد متغيّرات PASCAL في الذاكرة
======================================================================
تبحث في كل متغيّرات الجلسة عن:
  - مصفوفات تصنيفات (قيم 0/1)  -> مرشّحة لـ y_true
  - مصفوفات احتمالات (قيم بين 0 و 1، غير ثنائية)  -> مرشّحة لـ y_score
  - أي متغيّر اسمه يحتوي pascal
تطبع لك قائمة بالمرشّحين لتعرف الأسماء الفعلية.
"""
import numpy as np

def _arraylike(v):
    try:
        a = np.asarray(v, dtype=float).ravel()
        return a if a.size >= 5 and np.isfinite(a).all() else None
    except Exception:
        return None

print("=" * 60)
print("متغيّرات يحتوي اسمها على 'pascal':")
print("=" * 60)
for name, val in list(globals().items()):
    if name.startswith("_"):
        continue
    if "pascal" in name.lower():
        t = type(val).__name__
        a = _arraylike(val)
        extra = f" | shape={a.shape}, min={a.min():.3f}, max={a.max():.3f}" if a is not None else ""
        print(f"  {name:30s} ({t}){extra}")

print()
print("=" * 60)
print("مرشّحات y_true (مصفوفات ثنائية 0/1):")
print("=" * 60)
for name, val in list(globals().items()):
    if name.startswith("_"):
        continue
    a = _arraylike(val)
    if a is not None:
        u = np.unique(a)
        if set(u.tolist()).issubset({0.0, 1.0}) and len(u) == 2:
            print(f"  {name:30s} | n={a.size}, positives={int(a.sum())}")

print()
print("=" * 60)
print("مرشّحات y_score (احتمالات في [0,1] وغير ثنائية):")
print("=" * 60)
for name, val in list(globals().items()):
    if name.startswith("_"):
        continue
    a = _arraylike(val)
    if a is not None:
        u = np.unique(a)
        if a.min() >= 0.0 and a.max() <= 1.0 and len(u) > 2:
            print(f"  {name:30s} | n={a.size}, min={a.min():.3f}, "
                  f"max={a.max():.3f}, mean={a.mean():.3f}")

print()
print("=" * 60)
print("دوال/نماذج محتملة (للتنبؤ إن لزم):")
print("=" * 60)
for name, val in list(globals().items()):
    if name.startswith("_"):
        continue
    if any(k in name.lower() for k in ("model", "cnn", "clf", "net")) and hasattr(val, "__class__"):
        print(f"  {name:30s} ({type(val).__name__})")

In [ ]:
"""
خلية كشف شاملة — شغّلها بعد Run All
تكشف: النماذج، الـ DataFrames، الـ tensors/arrays الكبيرة، وأي شيء فيه pascal.
انسخ مخرجاتها وأرسلها لي.
"""
import numpy as np

print("=" * 55); print("النماذج (PyTorch / sklearn / keras):"); print("=" * 55)
for n, v in list(globals().items()):
    if n.startswith("_"): continue
    cls = type(v).__name__
    mod = type(v).__module__ or ""
    if any(k in mod for k in ("torch", "sklearn", "keras", "tensorflow")) and \
       any(h in dir(v) for h in ("forward", "predict", "predict_proba", "__call__")) and \
       not callable(v).__class__.__name__ == "builtin_function_or_method":
        if cls not in ("function", "type", "module"):
            print(f"  {n:28s} -> {cls}  (module: {mod.split('.')[0]})")

print("\n" + "=" * 55); print("DataFrames:"); print("=" * 55)
for n, v in list(globals().items()):
    if n.startswith("_"): continue
    if type(v).__name__ == "DataFrame":
        print(f"  {n:28s} shape={v.shape}  cols={list(v.columns)[:8]}")

print("\n" + "=" * 55); print("Tensors / arrays كبيرة (>=20 عنصر):"); print("=" * 55)
for n, v in list(globals().items()):
    if n.startswith("_"): continue
    t = type(v).__name__
    if t in ("ndarray", "Tensor"):
        try:
            shp = tuple(v.shape)
            size = int(np.prod(shp))
            if size >= 20:
                print(f"  {n:28s} {t} shape={shp}")
        except Exception:
            pass

print("\n" + "=" * 55); print("أي اسم فيه pascal / pcg / test / loader:"); print("=" * 55)
for n, v in list(globals().items()):
    if n.startswith("_"): continue
    if any(k in n.lower() for k in ("pascal", "pcg", "loader", "dataset", "_test", "test_")):
        print(f"  {n:28s} -> {type(v).__name__}")